# 도심 침수 예측 GNN — 데이터 전처리 파이프라인

**데이터**: 서울시 하수관로 수위 센서 + 도로노면 수위 센서  
**목표**: GNN 기반 침수 예측 모델 학습용 데이터셋 구축  
**공통 학습 기간**: 2024-01-01 ~ 2025-08-31 (20개월)  

| Step | 내용 | 주요 출력 |
|------|------|-----------|
| 01 | 원본 CSV/TXT → Parquet 변환 | `raw_parquet/` |
| 02 | 스키마 통일 및 통합 | `merged/` |
| 03 | 이상치 제거 + 결측치 보간 (병렬 처리) | `cleaned/` |
| 04 | 제원표 조인 (위경도, 관규격 등 메타데이터) | `sewer_node.parquet`, `road_node.parquet` |
| 05 | 탐색적 분석 + 교차 상관 분석 | `correlation_results.parquet` |
| 06 | 파생 피처 생성 | `features/sewer/`, `features/road/` |
| 07 | 인접행렬 생성 (Gaussian 가중치 σ=300m) | `adjacency_expanded.parquet` |
| 08 | 공통 기간 분리 (2024-01 ~ 2025-08) | `overlap/` |
| 09 | 정규화 (Train 파라미터만 사용, leakage 방지) | `*_normalized.parquet` |
| 10 | Train / Val / Test 시간 기반 분리 | `train/`, `val/`, `test/` |
| 11 | GNN 설정 파일 생성 | `gnn_config.json` |


## Step 01. 원본 CSV/TXT → Parquet 변환

- 하수관로 CSV 44개 (2022-01 ~ 2025-08) → `raw_parquet/sewer/`
- 도로노면 TXT 24개 (2024-01 ~ 2025-12) → `raw_parquet/road/`
- 인코딩 자동 감지 (`cp949` / `utf-8-sig` / `utf-8`), 한영 컬럼명 통합

> **NOTE**: 원본 파일이 `./sewer_csv/`, `./road_txt/` 경로에 있어야 합니다.

In [12]:
import pandas as pd
import glob, os

COLUMN_MAP = {
    '고유번호'  : 'sensor_id',  '?고유번호' : 'sensor_id',
    '측정일자'  : 'timestamp',  '측정수위'  : 'level',
    '통신상태'  : 'comm_status','unq_no'    : 'sensor_id',
    'msrmt_ymd' : 'timestamp',  'msrmt_watl': 'level',
    'sgn_stts'  : 'comm_status',
}

def read_sewer_file(f):
    for enc in ['cp949', 'utf-8-sig', 'utf-8']:
        try:
            df = pd.read_csv(f, encoding=enc)
            df.columns = [c.strip() for c in df.columns]
            df = df.rename(columns=COLUMN_MAP)
            if 'sensor_id' in df.columns and 'timestamp' in df.columns:
                return df, enc
        except Exception:
            continue
    return None, None

def save_parquet(df, out_path):
    df['timestamp']   = pd.to_datetime(df['timestamp']).astype('datetime64[ns]')
    df['sensor_id']   = df['sensor_id'].astype(str)
    df['level']       = pd.to_numeric(df['level'], errors='coerce')
    if 'comm_status' in df.columns:
        df['comm_status'] = df['comm_status'].astype(str)
    df.reset_index(drop=True).to_parquet(out_path, index=False, engine='pyarrow')

def convert_sewer(csv_dir):
    files = sorted(glob.glob(os.path.join(csv_dir, '*.csv')))
    os.makedirs('./dataset/processed/raw_parquet/sewer', exist_ok=True)
    for f in files:
        fname    = os.path.splitext(os.path.basename(f))[0]
        out_path = f'./dataset/processed/raw_parquet/sewer/{fname}.parquet'
        if os.path.exists(out_path):
            print(f'SKIP: {fname}'); continue
        df, enc = read_sewer_file(f)
        if df is None:
            print(f'ERROR: {fname}'); continue
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        save_parquet(df, out_path)
        print(f'OK [{enc}]: {fname} → {len(df):,}행')

def convert_road(txt_dir):
    files = sorted(glob.glob(os.path.join(txt_dir, '*.txt')))
    os.makedirs('./dataset/processed/raw_parquet/road', exist_ok=True)
    skip = ['2023년 1~12월.txt']
    for f in files:
        fname    = os.path.splitext(os.path.basename(f))[0]
        out_path = f'./dataset/processed/raw_parquet/road/{fname}.parquet'
        if os.path.basename(f) in skip:
            print(f'SKIP(부족): {fname}'); continue
        if os.path.exists(out_path):
            print(f'SKIP: {fname}'); continue
        try:
            df = pd.read_csv(f, sep='\t', encoding='cp949')
            df = df.rename(columns={'ROADGAUGE_NAME':'sensor_id',
                                    'DATA_TIME':'timestamp','LEVEL_DATA':'level'})
            df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            save_parquet(df, out_path)
            print(f'OK: {fname} → {len(df):,}행')
        except Exception as e:
            print(f'ERROR: {fname} → {e}')

convert_sewer('./sewer_csv/')
convert_road('./road_txt/')


## Step 02. 스키마 통일 및 통합

- 2025-07/08 파일의 영문 컬럼명(`se_cd`, `se_nm`, `pstn_info`)을 한글로 통일
- 기준 스키마 외 신규 컬럼(`위치정보`) 제거
- `ParquetWriter` 청크 방식 병합으로 메모리 절약


In [13]:
import glob, os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SEWER_RENAME = {
    'se_cd'    : '구분코드',
    'se_nm'    : '구분명',
    'pstn_info': '위치정보',  # 기준 스키마에 없으므로 제거됨
}

SEWER_BASE_COLS = ['sensor_id', '구분코드', '구분명', 'timestamp', 'level', 'comm_status']
ROAD_BASE_COLS  = ['sensor_id', 'timestamp', 'level']

SEWER_SCHEMA = pa.schema([
    ('sensor_id',  pa.string()),  ('구분코드', pa.string()),
    ('구분명',     pa.string()),  ('timestamp', pa.timestamp('ns')),
    ('level',      pa.float64()), ('comm_status', pa.string()),
])
ROAD_SCHEMA = pa.schema([
    ('sensor_id', pa.string()),  ('timestamp', pa.timestamp('ns')),
    ('level',     pa.float64()),
])

def align_schema(df, rename_map, base_cols):
    df = df.rename(columns=rename_map)
    for c in base_cols:
        if c not in df.columns: df[c] = None
    df = df[base_cols].copy()
    df['sensor_id'] = df['sensor_id'].astype(str)
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce').astype('datetime64[ns]')
    df['level']     = pd.to_numeric(df['level'], errors='coerce')
    if 'comm_status' in df.columns:
        df['comm_status'] = df['comm_status'].astype(str)
    if '구분코드' in df.columns:
        df['구분코드'] = df['구분코드'].astype(str)
    if '구분명' in df.columns:
        df['구분명'] = df['구분명'].astype(str)
    return df

def merge_parquets(raw_dir, out_path, rename_map, base_cols, schema):
    files  = sorted(glob.glob(os.path.join(raw_dir, '*.parquet')))
    writer = pq.ParquetWriter(out_path, schema)
    total  = 0
    for f in files:
        df    = pd.read_parquet(f)
        df    = align_schema(df, rename_map, base_cols)
        table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
        writer.write_table(table)
        total += len(df)
        print(f'  {os.path.basename(f)} → {len(df):,}행 (누적 {total:,}행)')
        del df, table
    writer.close()
    print(f'완료: {out_path} / 총 {total:,}행')

os.makedirs('./dataset/processed/merged', exist_ok=True)
merge_parquets('./dataset/processed/raw_parquet/sewer/',
               './dataset/processed/merged/sewer_all.parquet', SEWER_RENAME, SEWER_BASE_COLS, SEWER_SCHEMA)
merge_parquets('./dataset/processed/raw_parquet/road/',
               './dataset/processed/merged/road_all.parquet',  {}, ROAD_BASE_COLS, ROAD_SCHEMA)


  tv_swm_wal_mea_202501.parquet → 13,050,772행 (누적 13,050,772행)
  tv_swm_wal_mea_202502.parquet → 11,926,246행 (누적 24,977,018행)
  tv_swm_wal_mea_202503.parquet → 13,365,372행 (누적 38,342,390행)
  tv_swm_wal_mea_202504.parquet → 12,964,487행 (누적 51,306,877행)
  tv_swm_wal_mea_202505.parquet → 13,816,019행 (누적 65,122,896행)
  tv_swm_wal_mea_202506.parquet → 14,138,425행 (누적 79,261,321행)
  tv_swm_wal_mea_202507.parquet → 13,439,379행 (누적 92,700,700행)
  tv_swm_wal_mea_202508.parquet → 14,760,743행 (누적 107,461,443행)
  하수관로_수위_현황_202201.parquet → 10,344,880행 (누적 117,806,323행)
  하수관로_수위_현황_202202.parquet → 9,162,503행 (누적 126,968,826행)
  하수관로_수위_현황_202203.parquet → 10,288,216행 (누적 137,257,042행)
  하수관로_수위_현황_202204.parquet → 10,231,925행 (누적 147,488,967행)
  하수관로_수위_현황_202205.parquet → 10,643,325행 (누적 158,132,292행)
  하수관로_수위_현황_202206.parquet → 10,413,924행 (누적 168,546,216행)
  하수관로_수위_현황_202207.parquet → 10,576,592행 (누적 179,122,808행)
  하수관로_수위_현황_202208.parquet → 10,472,109행 (누적 189,594,917행)
  하수관로_수위_현황_202

## Step 03. 이상치 제거 + 결측치 보간 (병렬 처리)

- **이상치**: 도로 오류코드 `{312, 419, 999, 1000}`, 하수 음수/측정범위 초과 → NaN
- **보간**: 갭 ≤ 10분 → 선형 보간, 갭 > 10분 → NaN 유지 (과도한 보간 방지)
- **병렬화**: 하수관로 workers=2 (파일당 ~970MB), 도로노면 workers=4 (파일당 ~141MB)


In [14]:
import os, glob, gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import multiprocessing as mp

SEWER_RAW        = './dataset/processed/raw_parquet/sewer/'
ROAD_RAW         = './dataset/processed/raw_parquet/road/'
SEWER_CLEAN_DIR  = './dataset/processed/cleaned/sewer/'
ROAD_CLEAN_DIR   = './dataset/processed/cleaned/road/'
MERGED_DIR       = './dataset/processed/cleaned/'
GAP_LIMIT_MIN    = 10
ROAD_ERROR_CODES = {312, 419, 999, 1000}
WORKERS_SEWER    = 2
WORKERS_ROAD     = 4

SEWER_SCHEMA = pa.schema([('sensor_id',pa.string()),('timestamp',pa.timestamp('ns')),('level',pa.float64())])
ROAD_SCHEMA  = pa.schema([('sensor_id',pa.string()),('timestamp',pa.timestamp('ns')),('level',pa.float64())])

for d in [SEWER_CLEAN_DIR, ROAD_CLEAN_DIR, MERGED_DIR]:
    os.makedirs(d, exist_ok=True)

# 제원표에서 sensor별 측정범위 최댓값 로드
master = pd.read_excel('./dataset/processed/서울시 수위계(하수관로) 제원표_20260310.xlsx',
                       header=1, engine='openpyxl')
master.columns = [c.replace('\n',' ').strip() for c in master.columns]
max_map = (master[['수위계번호 (지점코드)','측정범위 최댓값(m)']]
           .rename(columns={'수위계번호 (지점코드)':'sid','측정범위 최댓값(m)':'mx'})
           .assign(sid=lambda d:d['sid'].astype(str),
                   mx=lambda d:pd.to_numeric(d['mx'],errors='coerce'))
           .dropna(subset=['mx']).set_index('sid')['mx'].to_dict())

# ── 보간 함수 ──────────────────────────────────────────────────────────────────
def _interpolate(df, gap_limit=GAP_LIMIT_MIN):
    parts = []
    for sid, grp in df.groupby('sensor_id', sort=False):
        grp = (grp[['timestamp','level']].sort_values('timestamp')
               .drop_duplicates('timestamp').set_index('timestamp'))
        grid = pd.date_range(grp.index.min(), grp.index.max(), freq='1min')
        grp  = grp.reindex(grid); grp.index.name = 'timestamp'
        grp['sensor_id'] = sid
        grp['level'] = grp['level'].interpolate(
            method='linear', limit=gap_limit,
            limit_direction='forward', limit_area='inside')
        parts.append(grp.reset_index()[['sensor_id','timestamp','level']])
    return pd.concat(parts, ignore_index=True)

# ── 파일 처리 함수 (fork 안전하도록 최상위 정의) ─────────────────────────────────
def process_sewer_file(args):
    f, _max_map, out_dir = args
    fname = os.path.splitext(os.path.basename(f))[0]
    out   = os.path.join(out_dir, f'{fname}.parquet')
    if os.path.exists(out): return f'SKIP: {fname}'
    try:
        df = pd.read_parquet(f, engine='pyarrow')
        neg  = int((df['level']<0).sum())
        df.loc[df['level']<0,'level'] = np.nan
        df['_max'] = df['sensor_id'].map(_max_map).fillna(20.0)
        over = int((df['level']>df['_max']).sum())
        df.loc[df['level']>df['_max'],'level'] = np.nan
        df.drop(columns=['_max'], inplace=True)
        df = _interpolate(df)
        df['timestamp'] = df['timestamp'].astype('datetime64[ns]')
        pq.write_table(pa.Table.from_pandas(df, schema=SEWER_SCHEMA, preserve_index=False), out)
        del df; gc.collect()
        return f'OK: {fname} [음수:{neg}, max초과:{over}]'
    except Exception as e: return f'ERROR: {fname} → {e}'

def process_road_file(args):
    f, out_dir = args
    fname = os.path.splitext(os.path.basename(f))[0]
    out   = os.path.join(out_dir, f'{fname}.parquet')
    if os.path.exists(out): return f'SKIP: {fname}'
    try:
        df = pd.read_parquet(f, engine='pyarrow')
        err = int(df['level'].isin(ROAD_ERROR_CODES).sum())
        df.loc[df['level'].isin(ROAD_ERROR_CODES),'level'] = np.nan
        df = _interpolate(df)
        df['timestamp'] = df['timestamp'].astype('datetime64[ns]')
        pq.write_table(pa.Table.from_pandas(df, schema=ROAD_SCHEMA, preserve_index=False), out)
        del df; gc.collect()
        return f'OK: {fname} [오류코드:{err}]'
    except Exception as e: return f'ERROR: {fname} → {e}'

def merge_cleaned(clean_dir, out_path, schema):
    files  = sorted(glob.glob(os.path.join(clean_dir,'*.parquet')))
    writer = pq.ParquetWriter(out_path, schema)
    total  = 0
    for f in files:
        table = pq.read_table(f, schema=schema)
        writer.write_table(table); total += table.num_rows; del table
    writer.close(); return total


In [15]:
# 하수관로 병렬 처리 (workers=2, 파일당 ~970MB)
ctx = mp.get_context('fork')
sewer_args = [(f, max_map, SEWER_CLEAN_DIR)
              for f in sorted(glob.glob(SEWER_RAW+'*.parquet'))]
print(f'하수관로 {len(sewer_args)}개 파일 처리 (workers={WORKERS_SEWER})')
with ctx.Pool(processes=WORKERS_SEWER, maxtasksperchild=1) as pool:
    for msg in pool.imap_unordered(process_sewer_file, sewer_args):
        print(f'  {msg}')

# 도로노면 병렬 처리 (workers=4, 파일당 ~141MB)
road_args = [(f, ROAD_CLEAN_DIR)
             for f in sorted(glob.glob(ROAD_RAW+'*.parquet'))]
print(f'\n도로노면 {len(road_args)}개 파일 처리 (workers={WORKERS_ROAD})')
with ctx.Pool(processes=WORKERS_ROAD, maxtasksperchild=1) as pool:
    for msg in pool.imap_unordered(process_road_file, road_args):
        print(f'  {msg}')

# 월별 파일 → 통합 parquet
n = merge_cleaned(SEWER_CLEAN_DIR, MERGED_DIR+'sewer_cleaned.parquet', SEWER_SCHEMA)
print(f'\nsewer_cleaned.parquet: {n:,}행')
n = merge_cleaned(ROAD_CLEAN_DIR,  MERGED_DIR+'road_cleaned.parquet',  ROAD_SCHEMA)
print(f'road_cleaned.parquet: {n:,}행')


하수관로 44개 파일 처리 (workers=2)
  SKIP: tv_swm_wal_mea_202502
  SKIP: tv_swm_wal_mea_202501
  SKIP: tv_swm_wal_mea_202503
  SKIP: tv_swm_wal_mea_202504

  SKIP: tv_swm_wal_mea_202504
  SKIP: tv_swm_wal_mea_202505  SKIP: tv_swm_wal_mea_202505
  SKIP: tv_swm_wal_mea_202506
  SKIP: tv_swm_wal_mea_202507
  SKIP: tv_swm_wal_mea_202508
  SKIP: tv_swm_wal_mea_202508
  SKIP: 하수관로_수위_현황_202201
  SKIP: 하수관로_수위_현황_202201  SKIP: 하수관로_수위_현황_202202
  SKIP: 하수관로_수위_현황_202202
  SKIP: 하수관로_수위_현황_202203
  SKIP: 하수관로_수위_현황_202204
  SKIP: 하수관로_수위_현황_202205
  SKIP: 하수관로_수위_현황_202205
  SKIP: 하수관로_수위_현황_202206
  SKIP: 하수관로_수위_현황_202206
  SKIP: 하수관로_수위_현황_202207
  SKIP: 하수관로_수위_현황_202208
  SKIP: 하수관로_수위_현황_202209

  SKIP: 하수관로_수위_현황_202209
  SKIP: 하수관로_수위_현황_202210
  SKIP: 하수관로_수위_현황_202210
  SKIP: 하수관로_수위_현황_202211
  SKIP: 하수관로_수위_현황_202212
  SKIP: 하수관로_수위_현황_202301

  SKIP: 하수관로_수위_현황_202301
  SKIP: 하수관로_수위_현황_202302
  SKIP: 하수관로_수위_현황_202302
  SKIP: 하수관로_수위_현황_202303
  SKIP: 하수관로_수위_현황_202304
  SKIP: 하수관로_수위_현황

## Step 04. 제원표 조인 (노드 메타데이터)

- 도로노면 `sensor_id`(지점명 형태)를 3단계 매핑으로 제원표 코드와 연결
  - 1단계: 완전 일치 / 2단계: 자치구 접두사 제거 후 일치 / 3단계: unmatched
- 관규격 파싱 (`Ø600`, `800X600` 등) → `pipe_height_m` (만수율 계산용)
- 출력: `sewer_node.parquet`, `road_node.parquet`


In [16]:
import re
import numpy as np
import pandas as pd
import os

OUT_DIR = './dataset/processed/cleaned/'

# ── 제원표 로드 ────────────────────────────────────────────────────────────────
road_master = pd.read_excel('./dataset/processed/서울시 수위계(도로) 제원표_20260310.xlsx',
                            header=1, engine='openpyxl')
road_master.columns = [c.replace('\n',' ').strip() for c in road_master.columns]
road_master = road_master.rename(columns={
    '수위계번호 (지점코드)':'sensor_code',
    '지점명':'지점명',
    '배수구역':'배수구역',
    '자치구':'자치구', 
    '위경도 lon.':'lat', 
    '위경도 lat.':'lon',
    '측정범위 최댓값(cm)':'max_level_cm', 
    '기왕 최대값(cm)':'hist_max_cm',
    '수위계 상태 (정상/미수신/정비중 등)':'status', 
    '자료관측주기 (1분/10분)':'obs_interval'
        }
    )
road_master['max_level_cm'] = pd.to_numeric(road_master['max_level_cm'], errors='coerce')

sewer_master = pd.read_excel('./dataset/processed/서울시 수위계(하수관로) 제원표_20260310.xlsx',
                             header=1, engine='openpyxl')
sewer_master.columns = [c.replace('\n',' ').strip() for c in sewer_master.columns]
sewer_master = sewer_master.rename(columns={
    '수위계번호 (지점코드)':'sensor_id', 
    '지점명':'지점명', 
    '배수구역':'배수구역',
    '자치구':'자치구', 
    '관규격':'관규격', 
    '위경도 lon.':'lat', 
    '위경도 lat.':'lon',
    '측정범위 최댓값(m)':'max_level_m', 
    '기왕 최대값(m)':'hist_max_m',
    '수위계 상태 (정상/미수신/정비중 등)':'status', 
    '자료관측주기 (1분/10분)':'obs_interval'
        }
    )
sewer_master['sensor_id']   = sewer_master['sensor_id'].astype(str)
sewer_master['max_level_m'] = pd.to_numeric(sewer_master['max_level_m'], errors='coerce')
sewer_master['lat'] = pd.to_numeric(sewer_master['lat'], errors='coerce')
sewer_master['lon'] = pd.to_numeric(sewer_master['lon'], errors='coerce')

# ── 관규격 파싱 → pipe_height_m ────────────────────────────────────────────────
def parse_pipe_height_m(spec):
    if pd.isna(spec): return np.nan
    s = str(spec).strip()
    m = re.match(r'[Øø](\d+)', s)
    if m: return float(m.group(1))/1000
    m = re.search(r'(\d+\.?\d*)X(\d+\.?\d*)', s, re.I)
    if m: return min(float(m.group(1)), float(m.group(2)))
    m = re.match(r'(\d+\.?\d*)$', s)
    if m: return float(m.group(1))
    return np.nan

sewer_master['pipe_height_m'] = sewer_master['관규격'].apply(parse_pipe_height_m)
print(f'관규격 파싱: {sewer_master["pipe_height_m"].notna().sum()}/{len(sewer_master)}개')


관규격 파싱: 438/629개


In [17]:
# ── sensor_id 매핑 (지점명 불일치 해결) ───────────────────────────────────────
def norm(s):         return re.sub(r'\s+','',str(s)).strip()
def strip_prefix(s): return re.sub(r'^[가-힣]+[시구군]\s+','',str(s)).strip()

m_exact = {norm(r['지점명']): r for _,r in road_master.iterrows()}
m_strip = {norm(strip_prefix(r['지점명'])): r for _,r in road_master.iterrows()}

road_pq_ids = (pd.read_parquet(OUT_DIR+'road_cleaned.parquet',
                               engine='pyarrow', columns=['sensor_id'])
               ['sensor_id'].unique())

mapping_rows, unmatched = [], []
for uid in road_pq_ids:
    n = norm(uid)
    if n in m_exact:
        row = m_exact[n].copy(); row['sensor_id']=uid; row['match_method']='exact'
        mapping_rows.append(row)
    elif n in m_strip:
        row = m_strip[n].copy(); row['sensor_id']=uid; row['match_method']='prefix_strip'
        mapping_rows.append(row)
    else:
        unmatched.append(uid)

road_map     = pd.DataFrame(mapping_rows)
road_missing = pd.DataFrame({'sensor_id':unmatched,'match_method':'unmatched'})
road_node    = pd.concat([road_map, road_missing], ignore_index=True)

print(f'정확 매핑:   {(road_node["match_method"]=="exact").sum()}개')
print(f'접두사 제거: {(road_node["match_method"]=="prefix_strip").sum()}개')
print(f'미매핑:      {(road_node["match_method"]=="unmatched").sum()}개')

NODE_COLS_SEWER = ['sensor_id',
                   '지점명',
                   '배수구역',
                   '자치구',
                   'lat',
                   'lon',
                   '관규격',
                   'pipe_height_m',
                   'max_level_m',
                   'hist_max_m',
                   'status',
                   'obs_interval'
                   ]

NODE_COLS_ROAD  = ['sensor_id',
                   'sensor_code',
                   '지점명',
                   '배수구역',
                   '자치구',
                   'lat',
                   'lon',
                   'max_level_cm',
                   'hist_max_cm',
                   'status',
                   'obs_interval',
                   'match_method'
                   ]

sewer_actual = set(pd.read_parquet(OUT_DIR+'sewer_cleaned.parquet',
                                   engine='pyarrow',columns=['sensor_id'])['sensor_id'].unique())
sewer_node = sewer_master[NODE_COLS_SEWER].query('sensor_id in @sewer_actual').reset_index(drop=True)
road_node  = road_node[[c for c in NODE_COLS_ROAD if c in road_node.columns]].copy()

sewer_node.to_parquet(OUT_DIR+'sewer_node.parquet', index=False, engine='pyarrow')
road_node.to_parquet(OUT_DIR+'road_node.parquet',   index=False, engine='pyarrow')
print(f'sewer_node: {len(sewer_node)}개 / road_node: {len(road_node)}개')


정확 매핑:   97개
접두사 제거: 16개
미매핑:      10개
sewer_node: 484개 / road_node: 123개


## Step 05. 탐색적 분석 + 상관 분석

- Parquet 필터 푸시다운으로 여름철(장마 6~9월) 데이터만 선택 로드
- 배수구역 + Haversine 거리 ≤ 1km → 센서 쌍 구성
- ±180분(18스텝) 범위 교차 상관(Cross-Correlation) 계산, 전체/이벤트(수위>0) 구분 (T_OUT=18 예측창과 정렬)
- 이벤트 상관은 연속축에서 shift 후 침수 시점만 평가 (마스크-후-shift 아티팩트 제거)


In [18]:
import numpy as np
import pandas as pd
import glob, os
from math import radians, sin, cos, sqrt, atan2

sewer_node = pd.read_parquet('./dataset/processed/cleaned/sewer_node.parquet', engine='pyarrow')
road_node  = pd.read_parquet('./dataset/processed/cleaned/road_node.parquet',  engine='pyarrow')

# Train 기간 내 여름철(장마) 데이터만 로드 — Parquet filter pushdown 활용
PERIODS = [(pd.Timestamp('2024-06-01'), pd.Timestamp('2024-10-01'))]

def load_period(path, s, e):
    return pd.read_parquet(path, engine='pyarrow',
                           filters=[('timestamp','>=',s),('timestamp','<',e)],
                           columns=['sensor_id','timestamp','level'])

sewer_raw = pd.concat([load_period('./dataset/processed/cleaned/sewer_cleaned.parquet',s,e) for s,e in PERIODS])
road_raw  = pd.concat([load_period('./dataset/processed/cleaned/road_cleaned.parquet', s,e) for s,e in PERIODS])

# 10분 집계 (하수: 평균 / 도로: 최대)
sewer_10 = (sewer_raw.set_index('timestamp').groupby('sensor_id')['level']
            .resample('10min').mean().reset_index().rename(columns={'level':'sewer_level'}))
road_10  = (road_raw.set_index('timestamp').groupby('sensor_id')['level']
            .resample('10min').max().reset_index().rename(columns={'level':'road_level'}))
print(f'sewer 10분: {len(sewer_10):,}행 / road 10분: {len(road_10):,}행')

# ── 센서 쌍 구성 (배수구역 일치 + 1km 이내) ───────────────────────────────────
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000; d = lambda x: radians(x)
    dlat, dlon = d(lat2-lat1), d(lon2-lon1)
    a = sin(dlat/2)**2 + cos(d(lat1))*cos(d(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

DIST_THRESHOLD_M = 1000
sewer_geo = sewer_node.dropna(subset=['lat','lon'])
road_geo  = road_node.dropna(subset=['lat','lon'])

pairs = []
for district, s_grp in sewer_geo.groupby('배수구역'):
    r_grp = road_geo[road_geo['배수구역']==district]
    if r_grp.empty: continue
    for _,sr in s_grp.iterrows():
        for _,rr in r_grp.iterrows():
            d = haversine(sr['lat'],sr['lon'],rr['lat'],rr['lon'])
            if d <= DIST_THRESHOLD_M:
                pairs.append({'sewer_id':sr['sensor_id'],'sewer_name':sr['지점명'],
                              'road_id':rr['sensor_id'],'road_name':rr['지점명'],
                              '배수구역':district,'자치구':sr['자치구'],
                              'distance_m':round(d,1),
                              'sewer_lat':sr['lat'],'sewer_lon':sr['lon'],
                              'road_lat':rr['lat'],'road_lon':rr['lon']})

pairs_df = pd.DataFrame(pairs).drop_duplicates(subset=['sewer_id','road_id'])
print(f'유효 센서 쌍: {len(pairs_df)}개')


sewer 10분: 5,389,345행 / road 10분: 1,018,247행
유효 센서 쌍: 765개


In [19]:
# ── 교차 상관 계산 ─────────────────────────────────────────────────────────────
MAX_LAG = 18; MIN_SAMPLES = 30   # ±180분: T_OUT=18 예측창과 정렬 (기존 6=±60분)

def cross_corr_best(x, y, max_lag=MAX_LAG, eval_mask=None):
    # eval_mask: 연속 시계열에서 shift한 뒤 평가할 시점만 선택 (이벤트 조건부 상관)
    #            → 마스크-후-shift 아티팩트 없이 진짜 시간 lag 유지
    best_lag, best_corr = 0, 0.0
    for lag in range(-max_lag, max_lag+1):
        xs   = x.shift(lag)
        mask = xs.notna() & y.notna()
        if eval_mask is not None: mask = mask & eval_mask
        if mask.sum() < MIN_SAMPLES: continue
        c = xs[mask].corr(y[mask])
        if abs(c) > abs(best_corr): best_corr, best_lag = c, lag
    base = x.notna() & y.notna()
    if eval_mask is not None: base = base & eval_mask
    return best_lag, best_corr, int(base.sum())

results = []
for _,row in pairs_df.iterrows():
    s_ts = sewer_10[sewer_10['sensor_id']==row['sewer_id']].set_index('timestamp')['sewer_level']
    r_ts = road_10[road_10['sensor_id']==row['road_id']].set_index('timestamp')['road_level']
    idx  = s_ts.index.union(r_ts.index)
    s_ts, r_ts = s_ts.reindex(idx), r_ts.reindex(idx)
    lag_all, corr_all, n_all = cross_corr_best(s_ts, r_ts)
    ev_mask = r_ts > 0
    if ev_mask.sum() >= MIN_SAMPLES:
        lag_ev, corr_ev, n_ev = cross_corr_best(s_ts, r_ts, eval_mask=ev_mask)
    else:
        lag_ev, corr_ev, n_ev = np.nan, np.nan, 0
    results.append({**row,
                    'corr_all':round(corr_all,4),'lag_all':lag_all,'n_all':n_all,
                    'corr_event':round(corr_ev,4) if not np.isnan(corr_ev) else np.nan,
                    'lag_event':lag_ev,'n_event':n_ev})

corr_df = pd.DataFrame(results)
corr_df.to_parquet('./dataset/processed/cleaned/correlation_results.parquet',
                   index=False, engine='pyarrow')
print(f'완료: {len(corr_df)}쌍 저장')
print(corr_df[['corr_all','corr_event']].describe().round(3))

# ── lag 검증: 하수관로가 도로보다 60분(6스텝) 넘게 선행하는 쌍이 있는가 ──
# 부호 규약: lag>0 = 하수관로가 도로보다 lag스텝(×10분) 선행 (예측에 유리한 방향)
for col, lcol, label in [('corr_all','lag_all','전체구간'),
                         ('corr_event','lag_event','이벤트구간')]:
    sub    = corr_df.dropna(subset=[col])
    strong = sub[sub[col].abs() >= 0.3]
    print(f'\n[{label}] |{col}|>=0.3 쌍: {len(strong)}/{len(sub)}')
    if len(strong):
        lead_short = ((strong[lcol] > 0) & (strong[lcol] <= 6)).sum()
        lead_long  =  (strong[lcol] > 6).sum()
        print(f'  하수 선행 <=60분 (0<lag<=6): {lead_short}개')
        print(f'  하수 선행  >60분 (lag>6)  : {lead_long}개  <- 기존 +-60분 창이 놓치던 영역')
        print(f'  {lcol} 분포:')
        print(strong[lcol].value_counts().sort_index().to_string())


/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encount

완료: 765쌍 저장
       corr_all  corr_event
count   765.000     174.000
mean      0.058      -0.000
std       0.154       0.292
min      -0.170      -0.743
25%       0.000      -0.124
50%       0.000       0.000
75%       0.000       0.213
max       0.716       0.648

[전체구간] |corr_all|>=0.3 쌍: 73/765
  하수 선행 <=60분 (0<lag<=6): 5개
  하수 선행  >60분 (lag>6)  : 0개  <- 기존 +-60분 창이 놓치던 영역
  lag_all 분포:
lag_all
-3     4
-2    17
-1    30
 0    17
 1     4
 6     1

[이벤트구간] |corr_event|>=0.3 쌍: 53/174
  하수 선행 <=60분 (0<lag<=6): 1개
  하수 선행  >60분 (lag>6)  : 0개  <- 기존 +-60분 창이 놓치던 영역
  lag_event 분포:
lag_event
-18.0     3
-17.0     1
-16.0     5
-13.0     1
-11.0     1
-9.0      1
-8.0      1
-5.0      1
-3.0     11
-2.0      9
-1.0     15
 0.0      3
 1.0      1


### Step 05-A. 변화율(level_diff) 기반 선행성 재검증

raw 수위는 도로 침수에 동조/후행으로 확인됨. 하수 **상승 속도**가 침수를 선행하는지 검증한다.

- 설계: 하수 `level_diff`(10분 변화율) vs 도로 `level` (도로측 불변 → 1변수만 변경)
- `fill_rate`는 level의 선형 스케일이라 Pearson lag가 raw와 동일 → 제외
- 결과는 `correlation_results_diff.parquet`에 별도 저장 (그래프 파이프라인 미영향)


In [20]:
# ── A안: 변화율(level_diff) 기반 선행성 재검증 ──────────────────────────────────
# 선행 의존: 앞의 센서쌍 구성 셀(sewer_10/road_10/pairs_df) + cross_corr_best 정의

# 하수 10분 변화율 (관별 시간순 diff)
sewer_10d = sewer_10.sort_values(['sensor_id','timestamp']).copy()
sewer_10d['sewer_diff'] = sewer_10d.groupby('sensor_id')['sewer_level'].diff()

results_d = []
for _, row in pairs_df.iterrows():
    s_ts = sewer_10d[sewer_10d['sensor_id']==row['sewer_id']].set_index('timestamp')['sewer_diff']
    r_ts = road_10[road_10['sensor_id']==row['road_id']].set_index('timestamp')['road_level']
    idx  = s_ts.index.union(r_ts.index)
    s_ts, r_ts = s_ts.reindex(idx), r_ts.reindex(idx)
    lag_all, corr_all, n_all = cross_corr_best(s_ts, r_ts)
    ev_mask = r_ts > 0
    if ev_mask.sum() >= MIN_SAMPLES:
        lag_ev, corr_ev, n_ev = cross_corr_best(s_ts, r_ts, eval_mask=ev_mask)
    else:
        lag_ev, corr_ev, n_ev = np.nan, np.nan, 0
    results_d.append({**row,
                      'corr_all':round(corr_all,4),'lag_all':lag_all,'n_all':n_all,
                      'corr_event':round(corr_ev,4) if not np.isnan(corr_ev) else np.nan,
                      'lag_event':lag_ev,'n_event':n_ev})

corr_d = pd.DataFrame(results_d)
corr_d.to_parquet('./dataset/processed/cleaned/correlation_results_diff.parquet',
                  index=False, engine='pyarrow')   # 별도 파일
print(f'[A안 변화율] 완료: {len(corr_d)}쌍 저장')
print(corr_d[['corr_all','corr_event']].describe().round(3))

# lag 검증 (raw 분석과 동일 진단 — 부호 규약: lag>0 = 하수 변화율이 도로보다 선행)
for col, lcol, label in [('corr_all','lag_all','전체구간'),
                         ('corr_event','lag_event','이벤트구간')]:
    sub    = corr_d.dropna(subset=[col])
    strong = sub[sub[col].abs() >= 0.3]
    print(f'\n[변화율·{label}] |{col}|>=0.3 쌍: {len(strong)}/{len(sub)}')
    if len(strong):
        lead_short = ((strong[lcol] > 0) & (strong[lcol] <= 6)).sum()
        lead_long  =  (strong[lcol] > 6).sum()
        print(f'  하수변화율 선행 <=60분 (0<lag<=6): {lead_short}개')
        print(f'  하수변화율 선행  >60분 (lag>6)  : {lead_long}개')
        print(f'  {lcol} 분포:')
        print(strong[lcol].value_counts().sort_index().to_string())
    else:
        print('  |상관|>=0.3 쌍 없음 (변화율은 차분이라 더 약할 수 있음)')


/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encount

[A안 변화율] 완료: 765쌍 저장
       corr_all  corr_event
count   765.000     174.000
mean      0.017      -0.032
std       0.079       0.268
min      -0.246      -0.679
25%       0.000      -0.246
50%       0.000       0.000
75%       0.000       0.051
max       0.491       0.644

[변화율·전체구간] |corr_all|>=0.3 쌍: 22/765
  하수변화율 선행 <=60분 (0<lag<=6): 0개
  하수변화율 선행  >60분 (lag>6)  : 0개
  lag_all 분포:
lag_all
-2     1
-1     7
 0    14

[변화율·이벤트구간] |corr_event|>=0.3 쌍: 57/174
  하수변화율 선행 <=60분 (0<lag<=6): 0개
  하수변화율 선행  >60분 (lag>6)  : 0개
  lag_event 분포:
lag_event
-18.0     3
-17.0     4
-15.0     3
-13.0     2
-11.0     2
-10.0     1
-9.0      1
-7.0      3
-3.0     14
-2.0     18
-1.0      5
 0.0      1


### Step 05-B. 관악 공동설치 쌍 교차상관 (GIS 최근접 페어링)

거리 1km 매칭 대신, GIS로 확인한 **도로 게이지의 최근접(공동설치) 하수 게이지**로 타이트 페어링.
- 부호: `lag>0` = 하수가 도로 침수보다 선행 (예측에 유리)
- 의존: Step 05의 `sewer_10`/`road_10`/`cross_corr_best`/`MIN_SAMPLES` 실행 후


In [21]:
# ── 관악 공동설치 쌍 (GIS 최근접 페어링) ──
GWANAK_PAIRS = [
    ('신림동 808-540', '21-0001'),  # 7m
    ('봉천로12길 10', '21-0013'),  # 49m
    ('신림동 513-14', '21-0008'),  # 73m
    ('방배동 442-2', '20-0001'),  # 109m
    ('조원로 5-6', '17-0014'),  # 170m
    ('봉천동 911-14', '21-0006'),  # 820m
]
gw_rows = []
for road_id, sewer_id in GWANAK_PAIRS:
    s_ts = sewer_10[sewer_10['sensor_id']==sewer_id].set_index('timestamp')['sewer_level']
    r_ts = road_10[road_10['sensor_id']==road_id].set_index('timestamp')['road_level']
    if len(s_ts)==0 or len(r_ts)==0:
        gw_rows.append({'road':road_id,'sewer':sewer_id,'corr_all':None,'lag_all':None,
                        'corr_event':None,'lag_event':None,'n_event':0,'note':'시계열없음'}); continue
    idx = s_ts.index.union(r_ts.index); s_ts, r_ts = s_ts.reindex(idx), r_ts.reindex(idx)
    lag_a, corr_a, n_a = cross_corr_best(s_ts, r_ts)
    ev = r_ts > 0
    if ev.sum() >= MIN_SAMPLES:
        lag_e, corr_e, n_e = cross_corr_best(s_ts, r_ts, eval_mask=ev)
    else:
        lag_e, corr_e, n_e = float('nan'), float('nan'), int(ev.sum())
    gw_rows.append({'road':road_id,'sewer':sewer_id,
                    'corr_all':round(corr_a,3),'lag_all':lag_a,
                    'corr_event':(round(corr_e,3) if corr_e==corr_e else None),
                    'lag_event':lag_e,'n_event':n_e,'note':''})
gw = pd.DataFrame(gw_rows)
print('=== 관악 공동설치 쌍 교차상관 (lag>0 = 하수 선행) ===')
print(gw.to_string(index=False))
_ev = gw[gw['lag_event'].notna()]
print(f"\n이벤트구간 평가가능 {len(_ev)}쌍 | 하수선행(lag>0): {(_ev['lag_event']>0).sum()} / 동시(0): {(_ev['lag_event']==0).sum()} / 후행(<0): {(_ev['lag_event']<0).sum()}")


=== 관악 공동설치 쌍 교차상관 (lag>0 = 하수 선행) ===
       road   sewer  corr_all  lag_all  corr_event  lag_event  n_event  note
신림동 808-540 21-0001     0.000      0.0         NaN        NaN        0      
  봉천로12길 10 21-0013       NaN      NaN         NaN        NaN        0 시계열없음
 신림동 513-14 21-0008     0.169     -1.0      -0.388       -5.0      116      
  방배동 442-2 20-0001     0.000      0.0         NaN        NaN       13      
    조원로 5-6 17-0014     0.000      0.0         NaN        NaN        0      
 봉천동 911-14 21-0006     0.000      0.0         NaN        NaN        0      

이벤트구간 평가가능 1쌍 | 하수선행(lag>0): 0 / 동시(0): 0 / 후행(<0): 1


/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/namjun/city_flood/.venv/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encount

In [24]:
import pandas as pd
BASE='./dataset/features/overlap/'
parts=[]
for p in ['train/road_train.parquet','val/road_val.parquet','test/road_test.parquet']:
    df=pd.read_parquet(BASE+p,columns=['sensor_id','timestamp','flood_flag'])
    parts.append(df[df['flood_flag']==1])
fl=pd.concat(parts); fl['timestamp']=pd.to_datetime(fl['timestamp']); fl['month']=fl['timestamp'].dt.month

print('■ 전체 도로 침수 — 월별 분포')
print(fl.groupby('month').size())

win=fl[fl['month'].isin([12,1,2])]
print(f"\n■ 겨울(12·1·2) 침수 {len(win):,} / 전체 {len(fl):,} ({len(win)/len(fl)*100:.1f}%)")

g=fl.groupby('sensor_id').size(); wg=win.groupby('sensor_id').size()
prof=pd.DataFrame({'total':g,'winter':wg}).fillna(0).astype(int)
prof['winter_pct']=(prof['winter']/prof['total']*100).round(0)
print(f"\n■ 침수 있는 센서 {len(prof)}개 중 겨울침수 있는 센서: {(prof['winter']>0).sum()}개")
print('\n■ 겨울 침수 많은 센서 상위 15 (이상 의심)')
print(prof.sort_values('winter',ascending=False).head(15).to_string())


■ 전체 도로 침수 — 월별 분포
month
1     295562
2     184628
3     224376
4     201314
5     162620
6     128924
7     322566
8     277147
9      96062
10     84417
11     81521
12     90219
dtype: int64

■ 겨울(12·1·2) 침수 570,409 / 전체 2,149,356 (26.5%)

■ 침수 있는 센서 84개 중 겨울침수 있는 센서: 30개

■ 겨울 침수 많은 센서 상위 15 (이상 의심)
               total  winter  winter_pct
sensor_id                               
개봉동 403-59    317868  119484        38.0
풍납동 403       153662   77453        50.0
시흥동 993       199947   66398        33.0
면목동 168-2     197641   57803        29.0
갈현동 466-9     102440   43470        42.0
통일로 446       146932   43096        29.0
화곡동 1094       44799   31662        71.0
월드컵로10길 40    206809   21669        10.0
월계동 9-2        45634   12115        27.0
봉천동 911-14     10290   10192        99.0
공릉동 608-2      20690   10061        49.0
서초대로70길 15      9936    9918       100.0
암사동 502-19     22834    9692        42.0
신영동 165        38791    9468        24.0
답십리동 488-230   36862    8389        23.

In [25]:
import pandas as pd
BASE='./dataset/features/overlap/'
parts=[]
for p in ['train/road_train.parquet','val/road_val.parquet','test/road_test.parquet']:
    df=pd.read_parquet(BASE+p,columns=['sensor_id','timestamp','level','flood_flag','flood_stage'])
    parts.append(df[df['flood_flag']==1])
fl=pd.concat(parts); fl['timestamp']=pd.to_datetime(fl['timestamp']); fl['month']=fl['timestamp'].dt.month
fl['season']=fl['month'].map(lambda m:'winter' if m in[12,1,2] else('summer' if m in[6,7,8,9] else 'other'))

print('■ 계절별 침수 level(cm) 분포')
print(fl.groupby('season')['level'].describe()[['count','mean','50%','max']].round(2))

print('\n■ 계절별 flood_stage 비율(%)  (1:0~5 2:5~20 3:20~50 4:50+cm)')
print((fl.groupby('season')['flood_stage'].value_counts(normalize=True).unstack().fillna(0)*100).round(1))

win=fl[fl['season']=='winter']
sus=win.groupby('sensor_id').size().sort_values(ascending=False).head(10).index
print('\n■ 겨울침수 상위센서 — 최빈 level 3개 점유율 (stuck 지표)')
for sid in sus:
    s=win[win['sensor_id']==sid]['level']; t3=s.value_counts().head(3)
    print(f"  {sid:16} n={len(s):6d}  top3={list(t3.index.round(1))} 점유={t3.sum()/len(s)*100:3.0f}%")


■ 계절별 침수 level(cm) 분포
           count  mean  50%   max
season                           
other   754248.0  1.51  1.0  96.0
summer  824699.0  1.25  1.0  96.0
winter  570409.0  1.64  1.0  41.0

■ 계절별 flood_stage 비율(%)  (1:0~5 2:5~20 3:20~50 4:50+cm)
flood_stage     1    2    3    4
season                          
other        93.4  6.3  0.3  0.0
summer       96.9  3.0  0.0  0.1
winter       92.2  6.9  0.9  0.0

■ 겨울침수 상위센서 — 최빈 level 3개 점유율 (stuck 지표)
  개봉동 403-59       n=119484  top3=[1.0, 6.0, 0.5] 점유=100%
  풍납동 403          n= 77453  top3=[1.0, 0.5, 0.699999988079071] 점유= 98%
  시흥동 993          n= 66398  top3=[1.0, 6.0, 0.5] 점유= 94%
  면목동 168-2        n= 57803  top3=[1.0, 6.0, 3.5] 점유=100%
  갈현동 466-9        n= 43470  top3=[1.0, 0.5, 0.30000001192092896] 점유= 84%
  통일로 446          n= 43096  top3=[1.0, 0.5, 0.6000000238418579] 점유= 99%
  화곡동 1094         n= 31662  top3=[1.0, 31.0, 26.0] 점유= 93%
  월드컵로10길 40       n= 21669  top3=[1.0, 0.0, 0.10000000149011612] 점유=100%
  월계동 9-2        

## 매핑 생성 코드(임시)

In [26]:
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R=6371000; d=lambda x: radians(x)
    a=sin(d(lat2-lat1)/2)**2 + cos(d(lat1))*cos(d(lat2))*sin(d(lon2-lon1)/2)**2
    return R*2*atan2(sqrt(a), sqrt(1-a))

ROAD_NODE='./dataset/processed/cleaned/road_node.parquet'
ROAD_IDX ='./dataset/features/road_node_index.parquet'
STATION  ='./dataset/processed/rain_station_coords.parquet'
OUT      ='./dataset/processed/road_rain_mapping.parquet'

rain_st   = pd.read_parquet(STATION)
road_node = pd.read_parquet(ROAD_NODE)
road_idx  = pd.read_parquet(ROAD_IDX)

# 학습에 쓰는 노드 + 좌표 있는 것만 (sewer 방식과 동일)
used = road_node[road_node['sensor_id'].isin(road_idx['sensor_id'])].dropna(subset=['lat','lon'])

rows=[]
for _, rn in used.iterrows():
    dists = rain_st.apply(lambda r: haversine(rn['lat'], rn['lon'], r['lat'], r['lon']), axis=1)
    best  = rain_st.loc[dists.idxmin()]
    rows.append({'sensor_id':rn['sensor_id'],
                 'rain_station_id':int(best['station_id']),
                 'rain_station_name':best['station_name'],
                 'distance_m':round(dists.min(),1)})
mapping = pd.DataFrame(rows)
mapping.to_parquet(OUT, index=False)

print(f'도로→강우 매핑 저장: {OUT}  ({len(mapping)}/{len(road_idx)}개)')
print('\n거리(m) 분포'); print(mapping['distance_m'].describe().round(0))
miss = set(road_idx['sensor_id']) - set(mapping['sensor_id'])
if miss: print(f'\n⚠️ 좌표없어 매핑안된 도로 {len(miss)}개:', list(miss)[:10])
print('\n샘플'); print(mapping.head(10).to_string(index=False))


도로→강우 매핑 저장: ./dataset/processed/road_rain_mapping.parquet  (90/90개)

거리(m) 분포
count      90.0
mean     1357.0
std       711.0
min        78.0
25%       732.0
50%      1450.0
75%      1937.0
max      2764.0
Name: distance_m, dtype: float64

샘플
 sensor_id  rain_station_id rain_station_name  distance_m
 가산동 60-24             2202              가산2P       601.9
 갈현동 466-9             1303              갈현1동      1104.4
 개봉동 307-1             2002              개봉2동       575.3
개봉동 403-59             2002              개봉2동        78.7
 개봉동 416-2             2002              개봉2동      1715.5
 공릉동 608-2              702               면목P      2257.4
  길동 386-1              201              강동구청      1529.4
  길동 459-4              201              강동구청      1397.8
논현동 164-11              101              강남구청      2354.1
 논현동 228-6              101              강남구청      1560.6


In [27]:
import pandas as pd, numpy as np

# 1) 강우: 관측소별 직전 6시간(36×10분) 누적
rain=pd.concat([pd.read_parquet(f'./dataset/features/rain/{s}/rain_{s}.parquet',
        columns=['station_id','timestamp','rainfall_mm']) for s in ['train','val','test']])
rain=rain.sort_values(['station_id','timestamp']).reset_index(drop=True)
rain['station_id']=rain['station_id'].astype(int)
rain['cum6h']=rain.groupby('station_id')['rainfall_mm'].rolling(36,min_periods=1).sum().reset_index(level=0,drop=True)
rain_lut=rain.set_index(['station_id','timestamp'])['cum6h']

# 2) 도로→관측소 매핑
s2st=pd.read_parquet('./dataset/processed/road_rain_mapping.parquet').set_index('sensor_id')['rain_station_id'].astype(int).to_dict()

# 3) 도로 침수 이벤트 + 직전강우 조인
parts=[]
for s in ['train','val','test']:
    df=pd.read_parquet(f'./dataset/features/overlap/{s}/road_{s}.parquet',columns=['sensor_id','timestamp','flood_flag'])
    parts.append(df[df['flood_flag']==1])
fl=pd.concat(parts,ignore_index=True)
ts=pd.to_datetime(fl['timestamp'])
fl['season']=ts.dt.month.map(lambda m:'winter' if m in[12,1,2] else('summer' if m in[6,7,8,9] else 'other'))
fl['station_id']=fl['sensor_id'].map(s2st)
fl['ts10']=ts.dt.floor('10min')
fl['cum6h']=rain_lut.reindex(pd.MultiIndex.from_arrays([fl['station_id'],fl['ts10']])).values

# 4) 계절별 직전6h 누적강우 분포
print('■ 계절별 침수시 직전6h 누적강우(mm)')
print(fl.groupby('season')['cum6h'].describe()[['count','mean','50%','75%','max']].round(2))
print('\n■ "비 없는 침수" 비율 (직전6h 강우 ≤ 임계)')
for thr in [0.0,0.5,1.0]:
    r=fl.assign(z=fl['cum6h'].fillna(0)<=thr).groupby('season')['z'].mean()*100
    print(f'  ≤{thr}mm : '+' / '.join(f'{k} {v:.1f}%' for k,v in r.items()))


■ 계절별 침수시 직전6h 누적강우(mm)
           count  mean  50%  75%    max
season                                 
other   569429.0  1.31  0.0  0.0   57.5
summer  626075.0  4.54  0.0  2.5  112.5
winter  469528.0  0.23  0.0  0.0   17.0

■ "비 없는 침수" 비율 (직전6h 강우 ≤ 임계)
  ≤0.0mm : other 84.3% / summer 73.7% / winter 90.2%
  ≤0.5mm : other 86.5% / summer 76.5% / winter 93.6%
  ≤1.0mm : other 88.1% / summer 77.9% / winter 95.2%


In [28]:
# 이전 셀의 rain_lut, s2st 재사용 (없으면 이전 셀 먼저 실행)
import pandas as pd
parts=[]
for s in ['train','val','test']:
    df=pd.read_parquet(f'./dataset/features/overlap/{s}/road_{s}.parquet',
                       columns=['sensor_id','timestamp','flood_flag','flood_stage'])
    parts.append(df[df['flood_flag']==1])
fl=pd.concat(parts,ignore_index=True)
ts=pd.to_datetime(fl['timestamp'])
fl['season']=ts.dt.month.map(lambda m:'winter' if m in[12,1,2] else('summer' if m in[6,7,8,9] else 'other'))
fl['station_id']=fl['sensor_id'].map(s2st)
fl['cum6h']=rain_lut.reindex(pd.MultiIndex.from_arrays([fl['station_id'],ts.dt.floor('10min')])).values

for label,sub in [('전체 level>0',fl),('stage≥2 (≥5cm)',fl[fl['flood_stage']>=2]),('stage≥3 (≥20cm)',fl[fl['flood_stage']>=3])]:
    g=sub.assign(rain=sub['cum6h'].fillna(0)>1.0).groupby('season').agg(건수=('rain','size'),비동반=('rain','mean'))
    g['비동반']=(g['비동반']*100).round(1)
    print(f'\n===== {label}: {len(sub):,}건 =====')
    print(g.rename(columns={'비동반':'비동반%'}).to_string())



===== 전체 level>0: 2,149,356건 =====
            건수  비동반%
season              
other   754248  11.9
summer  824699  22.1
winter  570409   4.8

===== stage≥2 (≥5cm): 119,642건 =====
           건수  비동반%
season             
other   49764   9.7
summer  25564  25.6
winter  44314   8.9

===== stage≥3 (≥20cm): 8,641건 =====
          건수  비동반%
season            
other   2198  51.4
summer  1219   2.0
winter  5224   9.4


In [30]:
import pandas as pd
parts=[]
for s in ['train','val','test']:
    df=pd.read_parquet(f'./dataset/features/overlap/{s}/road_{s}.parquet',columns=['sensor_id','timestamp','level','flood_flag'])
    parts.append(df[df['flood_flag']==1])
fl=pd.concat(parts,ignore_index=True)
fl['ts']=pd.to_datetime(fl['timestamp'])
fl['station_id']=fl['sensor_id'].map(s2st)
fl['cum6h']=rain_lut.reindex(pd.MultiIndex.from_arrays([fl['station_id'],fl['ts'].dt.floor('10min')])).values
fl['has_rain']=fl['cum6h'].fillna(0)>1     # 직전6h 강우 동반 여부

stuck=lambda s: s.value_counts().head(3).sum()/len(s)*100      # 최빈3값 점유%
g=fl.groupby('sensor_id').agg(floods=('level','size'),
                              stuck=('level',stuck),
                              rain=('has_rain',lambda x:x.mean()*100)).round(1)
bad=g[g['stuck']>=80]; good=g[g['stuck']<80]
tot=g['floods'].sum()
print(f'침수센서 {len(g)}개 / 총침수 {int(tot):,}')
print(f"■ stuck센서(최빈3값≥80%): {len(bad)}개, 침수 {int(bad.floods.sum()):,} ({bad.floods.sum()/tot*100:.1f}%)")
print(f"■ 정상센서: {len(good)}개, 침수 {int(good.floods.sum()):,} ({good.floods.sum()/tot*100:.1f}%)")

gf=fl[fl['sensor_id'].isin(good.index)]
print(f"   └ 정상센서 강우동반%(전체): {gf['has_rain'].mean()*100:.1f}%")
gs=gf[gf['ts'].dt.month.isin([6,7,8,9])]
print(f"   └ 정상센서 강우동반%(여름): {gs['has_rain'].mean()*100:.1f}%")
print('\n■ stuck센서 상위10'); print(bad.sort_values('floods',ascending=False).head(10).to_string())


침수센서 84개 / 총침수 2,149,356
■ stuck센서(최빈3값≥80%): 47개, 침수 2,057,449 (95.7%)
■ 정상센서: 37개, 침수 91,907 (4.3%)
   └ 정상센서 강우동반%(전체): 15.6%
   └ 정상센서 강우동반%(여름): 53.5%

■ stuck센서 상위10
            floods      stuck  rain
sensor_id                          
개봉동 403-59  317868  99.300003   8.1
월드컵로10길 40  206809  99.900002   9.2
시흥동 993     199947  94.199997   6.9
면목동 168-2   197641  98.599998  16.3
풍납동 403     153662  96.000000  14.2
통일로 446     146932  97.500000   0.0
갈현동 466-9   102440  92.099998   6.2
강일동 137-10   75251  98.300003   0.0
송파동 195-7    48130  90.900002  21.4
서초동 1396     48129  90.599998   0.0


In [31]:
import pandas as pd
SID='개봉동 403-59'   # stuck 상위 (원하면 봉천동 911-14로도)
raw=pd.read_parquet('./dataset/processed/raw_parquet/road/2024년 1월.parquet')
raw=raw[raw['sensor_id']==SID].sort_values('timestamp')
print(f'■ RAW {SID} 2024-01: {len(raw):,}행')
gap=raw['timestamp'].diff().dt.total_seconds().div(60)
print('  읽기 간격(분): ', gap.describe()[['min','50%','max']].round(1).to_dict())
print('  level 최빈값:'); print(raw['level'].value_counts().head(8))
print('  level describe:', raw['level'].describe()[['min','mean','50%','max']].round(2).to_dict())

cln=pd.read_parquet('./dataset/processed/cleaned/road/2024년 1월.parquet')
cln=cln[cln['sensor_id']==SID]
print(f'\n■ CLEANED {SID} 2024-01: {len(cln):,}행 (1분그리드+보간)')
print('  level 최빈값:'); print(cln['level'].value_counts().head(8))


■ RAW 개봉동 403-59 2024-01: 44,626행
  읽기 간격(분):  {'min': 0.8, '50%': 1.0, 'max': 3.9}
  level 최빈값:
level
1     25186
6     11192
0      8246
11        2
Name: count, dtype: int64
  level describe: {'min': 0.0, 'mean': 2.07, '50%': 1.0, 'max': 11.0}

■ CLEANED 개봉동 403-59 2024-01: 44,639행 (1분그리드+보간)
  level 최빈값:
level
1.000000    19133
6.000000     6758
0.000000     6581
0.500000       14
0.750000        6
0.250000        6
0.411765        5
0.470588        5
Name: count, dtype: int64


In [32]:
# 이전 셀의 rain_lut, s2st 재사용
import pandas as pd

MARGIN   = 5.0    # baseline 대비 최소 상승(cm) — 튜닝 가능
RAIN_THR = 1.0    # 직전6h 누적강우 최소(mm)

# 1) 전체 도로 level 로드
road=pd.concat([pd.read_parquet(f'./dataset/features/overlap/{s}/road_{s}.parquet',
                columns=['sensor_id','timestamp','level']) for s in ['train','val','test']],
               ignore_index=True).dropna(subset=['level'])

# 2) 센서별 baseline = 최빈값(평소 마른 수위)
base=road.groupby('sensor_id')['level'].agg(baseline=lambda x:x.mode().iloc[0], median='median').round(2)
print('■ 센서별 baseline(최빈값) 분포'); print(base['baseline'].describe().round(2).to_dict())
print('\n■ baseline 높은 센서 상위10 (오프셋/stuck 의심)')
print(base.sort_values('baseline',ascending=False).head(10).to_string())

# 3) 후보 침수 = level ≥ baseline + MARGIN
road=road.merge(base['baseline'], on='sensor_id')
cand=road[road['level'] >= road['baseline']+MARGIN].copy()

# 4) 후보에 강우 조인 → 진짜 침수 = 후보 ∧ 강우
cand['ts']=pd.to_datetime(cand['timestamp']); cand['station_id']=cand['sensor_id'].map(s2st)
cand['cum6h']=rain_lut.reindex(pd.MultiIndex.from_arrays([cand['station_id'],cand['ts'].dt.floor('10min')])).values
cand['real']=cand['cum6h'].fillna(0)>=RAIN_THR
rf=cand[cand['real']]

print(f"\n기존 flood_flag(level>0) : {(road['level']>0).sum():,}")
print(f"후보(level≥baseline+{MARGIN}): {len(cand):,}")
print(f"진짜 침수(후보 ∧ 강우)   : {len(rf):,}")
print(f"진짜 침수 보유 센서       : {rf['sensor_id'].nunique()} / {road['sensor_id'].nunique()}개")
neg=len(road)-len(rf)
print(f"진짜 불균형 비율          : 1:{neg/max(len(rf),1):.0f}")
print('\n■ 진짜 침수 월별 분포 (여름 집중이어야 정상)')
print(rf.groupby(rf['ts'].dt.month).size().to_string())


■ 센서별 baseline(최빈값) 분포
{'count': 100.0, 'mean': 0.04, 'std': 0.2, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}

■ baseline 높은 센서 상위10 (오프셋/stuck 의심)
             baseline  median
sensor_id                    
개봉동 403-59        1.0     1.0
월드컵로10길 40        1.0     1.0
오류동 135-4         1.0     1.0
시흥동 1001-16       1.0     1.0
갈월동 69-63         0.0     0.0
갈현동 466-9         0.0     0.0
개봉동 173-3         0.0     0.0
강일동 137-10        0.0     0.0
개봉동 307-1         0.0     0.0
개봉동 353-34        0.0     0.0

기존 flood_flag(level>0) : 2,149,356
후보(level≥baseline+5.0): 120,835
진짜 침수(후보 ∧ 강우)   : 17,473
진짜 침수 보유 센서       : 22 / 100개
진짜 불균형 비율          : 1:1297

■ 진짜 침수 월별 분포 (여름 집중이어야 정상)
ts
1     2418
2     1996
3     4177
4      937
5      393
6      925
7     5354
8      634
9        5
12     634


In [35]:
import pandas as pd

print("===== 하수관로 (단위 m) — 월별 level 분포 =====")
for m in ['202401','202407','202408','202409']:
    p=pd.read_parquet(f'./dataset/processed/raw_parquet/sewer/하수관로_수위_현황_{m}.parquet',
                      columns=['level'])
    d=p['level'].describe(percentiles=[.5,.95,.99])
    print(f"  {m}: mean={d['mean']:.3f} 50%={d['50%']:.2f} "
          f"95%={d['95%']:.2f} 99%={d['99%']:.2f} max={d['max']:.2f}")

print("\n===== 도로노면 (단위 cm) — 월별 level 분포 =====")
for m,f in [('2024-01','2024년 1월'),('2024-07','2024년 7월'),
            ('2024-08','2024년 8월'),('2024-09','2024년 9월')]:
    p=pd.read_parquet(f'./dataset/processed/raw_parquet/road/{f}.parquet',columns=['level'])
    d=p['level'].describe(percentiles=[.5,.95,.99])
    nz=(p['level']>0).mean()*100
    print(f"  {m}: mean={d['mean']:.2f} 50%={d['50%']:.1f} 95%={d['95%']:.1f} "
          f"99%={d['99%']:.1f} max={d['max']:.1f}  | level>0 비율 {nz:.1f}%")


===== 하수관로 (단위 m) — 월별 level 분포 =====
  202401: mean=0.148 50%=0.10 95%=0.41 99%=0.93 max=3.58
  202407: mean=0.158 50%=0.11 95%=0.43 99%=0.77 max=11.20
  202408: mean=0.142 50%=0.10 95%=0.40 99%=0.67 max=3.33
  202409: mean=0.150 50%=0.11 95%=0.40 99%=0.70 max=3.52

===== 도로노면 (단위 cm) — 월별 level 분포 =====
  2024-01: mean=0.23 50%=0.0 95%=1.0 99%=6.0 max=26.0  | level>0 비율 11.4%
  2024-07: mean=0.16 50%=0.0 95%=1.0 99%=1.0 max=1000.0  | level>0 비율 13.3%
  2024-08: mean=0.12 50%=0.0 95%=1.0 99%=1.0 max=1000.0  | level>0 비율 9.6%
  2024-09: mean=0.09 50%=0.0 95%=1.0 99%=1.0 max=31.0  | level>0 비율 9.3%


In [36]:
import pandas as pd

raw = pd.read_parquet('./dataset/processed/raw_parquet/sewer/하수관로_수위_현황_202407.parquet',
                      columns=['sensor_id','level','comm_status'])

print('■ 통신상태 분포'); print(raw['comm_status'].value_counts())
print('\n■ 통신상태별 level(m) 분포 (2024-07)')
print(raw.groupby('comm_status')['level'].describe()[['count','mean','50%','max']].round(2).to_string())

# 통신양호만 봤을 때 max
good = raw[raw['comm_status'].str.contains('양호', na=False)]
print(f"\n통신양호 level: max={good['level'].max():.2f}, 99%={good['level'].quantile(.99):.2f}")
print(f"전체      level: max={raw['level'].max():.2f}")


■ 통신상태 분포
comm_status
통신양호    13265236
Name: count, dtype: int64

■ 통신상태별 level(m) 분포 (2024-07)
                  count  mean   50%   max
comm_status                              
통신양호         13265236.0  0.16  0.11  11.2

통신양호 level: max=11.20, 99%=0.77
전체      level: max=11.20


In [38]:
import pandas as pd

# ── 원본 7월 ──
ORIG='dataset/processed/하수관로_수위_현황_202407.csv'   # 경로 수정
o=pd.read_csv(ORIG, encoding='cp949'); o.columns=[c.strip() for c in o.columns]
o['측정수위']=pd.to_numeric(o['측정수위'], errors='coerce')
print('원본 측정수위: max=%.2f  99%%=%.2f' % (o['측정수위'].max(), o['측정수위'].quantile(.99)))
print('원본 최댓값 행:')
print(o.loc[o['측정수위'].idxmax(), ['?고유번호','측정일자','측정수위','통신상태']].to_dict())

# ── parquet 7월 ──
p=pd.read_parquet('./dataset/processed/raw_parquet/sewer/하수관로_수위_현황_202407.parquet')
print('\nparquet level : max=%.2f' % p['level'].max())
top=p.loc[p['level'].idxmax()]
print('parquet 최댓값 센서/시각:', top['sensor_id'], str(top['timestamp']))
# 그 센서의 원본 값과 대조
sid=top['sensor_id']
print(f'\n[{sid}] parquet level 상위5:', p[p['sensor_id']==sid]['level'].nlargest(5).tolist())


원본 측정수위: max=11.20  99%=0.77
원본 최댓값 행:
{'?고유번호': '07-0009', '측정일자': '2024-07-04 15:48:00.0', '측정수위': 11.2, '통신상태': '통신양호'}

parquet level : max=11.20
parquet 최댓값 센서/시각: 07-0009 2024-07-04 15:48:00

[07-0009] parquet level 상위5: [11.2, 11.2, 11.2, 11.2, 11.2]


## Step 06. 파생 Feature 생성

- **하수관로** (8개): `level_diff`, `fill_rate`(만수율), `hour/month/is_weekend`
- **도로노면** (9개): `level_diff`, `flood_flag`, `flood_stage`, `hour/month/is_weekend`
- `flood_stage`: 0=정상, 1=0~5cm, 2=5~20cm, 3=20~50cm, 4=50cm 초과
- 월별 파일 단위 처리로 메모리 사용 최소화


In [ ]:
import pandas as pd, numpy as np, glob, os, gc
import pyarrow as pa, pyarrow.parquet as pq

sewer_node = pd.read_parquet('./dataset/processed/cleaned/sewer_node.parquet', engine='pyarrow')
road_node  = pd.read_parquet('./dataset/processed/cleaned/road_node.parquet',  engine='pyarrow')

pipe_map  = sewer_node.dropna(subset=['pipe_height_m']).set_index('sensor_id')['pipe_height_m'].to_dict()
maxlv_map = sewer_node.dropna(subset=['max_level_m']).set_index('sensor_id')['max_level_m'].to_dict()

SEWER_FEAT_SCHEMA = pa.schema([
    ('sensor_id',pa.string()), ('timestamp',pa.timestamp('ns')),
    ('level',pa.float64()),    ('level_diff',pa.float64()),
    ('fill_rate',pa.float64()),('hour',pa.int8()),
    ('month',pa.int8()),       ('is_weekend',pa.int8())])
ROAD_FEAT_SCHEMA = pa.schema([
    ('sensor_id',pa.string()), ('timestamp',pa.timestamp('ns')),
    ('level',pa.float64()),    ('level_diff',pa.float64()),
    ('flood_flag',pa.int8()),  ('flood_stage',pa.int8()),
    ('hour',pa.int8()),        ('month',pa.int8()),
    ('is_weekend',pa.int8())])

os.makedirs('./dataset/features/sewer/', exist_ok=True)
os.makedirs('./dataset/features/road/',  exist_ok=True)

# ── 하수관로 ───────────────────────────────────────────────────────────────────
for f in sorted(glob.glob('./dataset/processed/cleaned/sewer/*.parquet')):
    fname = os.path.basename(f)
    out   = f'./dataset/features/sewer/{fname}'
    if os.path.exists(out): print(f'SKIP: {fname}'); continue
    df = pd.read_parquet(f, engine='pyarrow').sort_values(['sensor_id','timestamp'])
    df['level_diff'] = df.groupby('sensor_id')['level'].diff().fillna(0)
    df['_d']         = df['sensor_id'].map(pipe_map).fillna(df['sensor_id'].map(maxlv_map))
    df['fill_rate']  = (df['level']/df['_d']).clip(0,1.0).astype('float64')
    df.drop(columns=['_d'], inplace=True)
    df['hour']       = df['timestamp'].dt.hour.astype('int8')
    df['month']      = df['timestamp'].dt.month.astype('int8')
    df['is_weekend'] = (df['timestamp'].dt.dayofweek>=5).astype('int8')
    df['level_diff'] = df['level_diff'].astype('float64')
    pq.write_table(pa.Table.from_pandas(df[SEWER_FEAT_SCHEMA.names],
                   schema=SEWER_FEAT_SCHEMA, preserve_index=False), out)
    print(f'OK: {fname} → {len(df):,}행'); del df; gc.collect()

# ── 도로노면 ───────────────────────────────────────────────────────────────────
for f in sorted(glob.glob('./dataset/processed/cleaned/road/*.parquet')):
    fname = os.path.basename(f)
    out   = f'./dataset/features/road/{fname}'
    if os.path.exists(out): print(f'SKIP: {fname}'); continue
    df = pd.read_parquet(f, engine='pyarrow').sort_values(['sensor_id','timestamp'])
    df['level_diff']  = df.groupby('sensor_id')['level'].diff().fillna(0)
    df['flood_flag']  = (df['level']>0).astype('int8')
    df['flood_stage'] = pd.cut(df['level'].fillna(0), bins=[-1,0,5,20,50,9999],
                               labels=[0,1,2,3,4]).astype('int8')
    df['hour']        = df['timestamp'].dt.hour.astype('int8')
    df['month']       = df['timestamp'].dt.month.astype('int8')
    df['is_weekend']  = (df['timestamp'].dt.dayofweek>=5).astype('int8')
    df['level_diff']  = df['level_diff'].astype('float64')
    pq.write_table(pa.Table.from_pandas(df[ROAD_FEAT_SCHEMA.names],
                   schema=ROAD_FEAT_SCHEMA, preserve_index=False), out)
    print(f'OK: {fname} → {len(df):,}행'); del df; gc.collect()


In [41]:
# ── 한글 폰트 ──
import matplotlib; matplotlib.use('Agg')
import matplotlib.font_manager as fm
cands=['NanumGothic','NanumBarunGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR','UnDotum','Baekmuk Gulim']
avail={f.name for f in fm.fontManager.ttflist}
korf=next((c for c in cands if c in avail),None)
if korf: matplotlib.rcParams['font.family']=korf; print('한글폰트:',korf)
else: print('한글폰트 미설치 → sudo apt install -y fonts-nanum; rm -rf ~/.cache/matplotlib')
matplotlib.rcParams['axes.unicode_minus']=False
import matplotlib.pyplot as plt

import pandas as pd, numpy as np, glob, os
os.makedirs('./dataset/processed/viz', exist_ok=True)

DOMAINS={
 'Sewer (m)': ('./dataset/processed/raw_parquet/sewer/*.parquet', (0,12), 120),
 'Road (cm)': ('./dataset/processed/raw_parquet/road/*.parquet',  (0,1010),202),
}

def aggregate(pattern, hrange, hbins):
    files=sorted(glob.glob(pattern)); print(f'  파일 {len(files)}개')
    edges=np.linspace(*hrange,hbins+1); lvl_hist=np.zeros(hbins)
    gap_edges=np.arange(0,61,1); gap_hist=np.zeros(len(gap_edges)-1)
    aggs=[]; vcs=[]
    for i,f in enumerate(files):
        df=pd.read_parquet(f,columns=['sensor_id','timestamp','level'])
        df['timestamp']=pd.to_datetime(df['timestamp'])
        h,_=np.histogram(df['level'].dropna().clip(*hrange),bins=edges); lvl_hist+=h
        aggs.append(df.groupby('sensor_id').agg(n=('level','size'),mx=('level','max'),
                                                t0=('timestamp','min'),t1=('timestamp','max')))
        vc=df.assign(lr=df['level'].round(2)).groupby('sensor_id')['lr'].value_counts().rename('c').reset_index()
        vcs.append(vc)
        s=df.sort_values(['sensor_id','timestamp'])
        gap=s.groupby('sensor_id')['timestamp'].diff().dt.total_seconds()/60
        gh,_=np.histogram(gap.dropna().clip(0,60),bins=gap_edges); gap_hist+=gh
        print(f'    [{i+1}/{len(files)}] {os.path.basename(f)}')
    big=pd.concat(aggs)
    st=big.groupby(level=0).agg(n=('n','sum'),mx=('mx','max'),t0=('t0','min'),t1=('t1','max'))
    span=(st['t1']-st['t0']).dt.total_seconds()/60+1
    st['missing_pct']=(1-st['n']/span).clip(lower=0)*100
    vc_all=pd.concat(vcs).groupby(['sensor_id','lr'])['c'].sum()
    st['stuck_pct']=vc_all.groupby(level=0).apply(lambda s:s.nlargest(3).sum()/s.sum()*100)
    return st, lvl_hist, edges, gap_hist, gap_edges

print('집계 시작 (하수 44개월 → 수 분 소요)')
results={name:aggregate(p,hr,hb) for name,(p,hr,hb) in DOMAINS.items()}

# ===== 그림 1: 이상치 =====
fig,ax=plt.subplots(2,2,figsize=(14,9))
for i,(name,(st,lh,ed,gh,ge)) in enumerate(results.items()):
    c=(ed[:-1]+ed[1:])/2
    ax[i,0].bar(c,lh,width=ed[1]-ed[0],color='steelblue'); ax[i,0].set_yscale('log')
    ax[i,0].set_title(f'{name} level 분포 (전체기간, log y)'); ax[i,0].set_xlabel('level'); ax[i,0].set_ylabel('건수(log)')
    ax[i,1].scatter(st['stuck_pct'],st['mx'],s=15,alpha=.5,color='crimson')
    ax[i,1].axvline(80,ls='--',c='gray',label='stuck 80%')
    ax[i,1].set_title(f'{name} 센서별 stuck% vs 최댓값'); ax[i,1].set_xlabel('최빈3값 점유율(%)'); ax[i,1].set_ylabel('최댓값'); ax[i,1].legend()
fig.suptitle('이상치 (전체 데이터셋)',fontsize=14); plt.tight_layout()
plt.savefig('./dataset/processed/viz/viz_outliers.png',dpi=110); plt.close()

# ===== 그림 2: 결측치 =====
fig,ax=plt.subplots(2,2,figsize=(14,9))
for i,(name,(st,lh,ed,gh,ge)) in enumerate(results.items()):
    ax[i,0].hist(st['missing_pct'].clip(0,100),bins=30,color='darkorange')
    ax[i,0].set_title(f'{name} 센서별 결측률'); ax[i,0].set_xlabel('결측 %'); ax[i,0].set_ylabel('센서 수')
    gc=(ge[:-1]+ge[1:])/2
    ax[i,1].bar(gc,gh,width=1,color='seagreen')
    ax[i,1].set_title(f'{name} 읽기 간격 분포 (분, <60)'); ax[i,1].set_xlabel('읽기 간격(분)'); ax[i,1].set_ylabel('건수')
fig.suptitle('결측치 (전체 데이터셋)',fontsize=14); plt.tight_layout()
plt.savefig('./dataset/processed/viz/viz_missing.png',dpi=110); plt.close()

print('\n저장: viz_outliers.png, viz_missing.png')
for name,(st,*_ ) in results.items():
    print(f"[{name}] 센서 {len(st)} | stuck≥80%: {(st['stuck_pct']>=80).sum()} | 결측>10%: {(st['missing_pct']>10).sum()} | max상위5: {st['mx'].nlargest(5).round(2).tolist()}")


한글폰트: NanumGothic
집계 시작 (하수 44개월 → 수 분 소요)
  파일 44개
    [1/44] tv_swm_wal_mea_202501.parquet
    [2/44] tv_swm_wal_mea_202502.parquet
    [3/44] tv_swm_wal_mea_202503.parquet
    [4/44] tv_swm_wal_mea_202504.parquet
    [5/44] tv_swm_wal_mea_202505.parquet
    [6/44] tv_swm_wal_mea_202506.parquet
    [7/44] tv_swm_wal_mea_202507.parquet
    [8/44] tv_swm_wal_mea_202508.parquet
    [9/44] 하수관로_수위_현황_202201.parquet
    [10/44] 하수관로_수위_현황_202202.parquet
    [11/44] 하수관로_수위_현황_202203.parquet
    [12/44] 하수관로_수위_현황_202204.parquet
    [13/44] 하수관로_수위_현황_202205.parquet
    [14/44] 하수관로_수위_현황_202206.parquet
    [15/44] 하수관로_수위_현황_202207.parquet
    [16/44] 하수관로_수위_현황_202208.parquet
    [17/44] 하수관로_수위_현황_202209.parquet
    [18/44] 하수관로_수위_현황_202210.parquet
    [19/44] 하수관로_수위_현황_202211.parquet
    [20/44] 하수관로_수위_현황_202212.parquet
    [21/44] 하수관로_수위_현황_202301.parquet
    [22/44] 하수관로_수위_현황_202302.parquet
    [23/44] 하수관로_수위_현황_202303.parquet
    [24/44] 하수관로_수위_현황_202304.parquet
    [25/44] 하

In [42]:
# ── 한글 폰트 ──
import matplotlib; matplotlib.use('Agg')
import matplotlib.font_manager as fm
cands=['NanumGothic','NanumBarunGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR','UnDotum','Baekmuk Gulim']
korf=next((c for c in cands if c in {f.name for f in fm.fontManager.ttflist}),None)
if korf: matplotlib.rcParams['font.family']=korf; print('한글폰트:',korf)
else: print('한글폰트 미설치 → sudo apt install -y fonts-nanum; rm -rf ~/.cache/matplotlib')
matplotlib.rcParams['axes.unicode_minus']=False
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, glob, os
os.makedirs('./dataset/processed/viz', exist_ok=True)

SEWER='./dataset/processed/raw_parquet/sewer/*.parquet'
ROAD ='./dataset/processed/raw_parquet/road/*.parquet'

# ============ 1. stuck 센서 시계열 ============
STUCK = {  # 변경 가능 (앞 분석의 stuck 상위 센서)
  ROAD : ['개봉동 403-59','봉천동 911-14','월드컵로10길 40','시흥동 993'],
  SEWER: ['07-0009'],
}
def load_series(pattern, sensors, freq='1h'):
    out={s:[] for s in sensors}
    for f in sorted(glob.glob(pattern)):
        df=pd.read_parquet(f,columns=['sensor_id','timestamp','level'])
        df=df[df['sensor_id'].isin(sensors)]
        if len(df)==0: continue
        df['timestamp']=pd.to_datetime(df['timestamp'])
        for s,g in df.groupby('sensor_id'):
            out[s].append(g.set_index('timestamp')['level'].resample(freq).max())
    return {s:(pd.concat(v).sort_index() if v else pd.Series(dtype=float)) for s,v in out.items()}

series={}
for pat,sl in STUCK.items(): series.update(load_series(pat,sl))
n=len(series); fig,ax=plt.subplots(n,1,figsize=(14,2.6*n),squeeze=False)
for i,(s,ts) in enumerate(series.items()):
    ax[i,0].plot(ts.index,ts.values,lw=0.6,color='crimson')
    ax[i,0].set_title(f'{s}  (시간별 max) — 평평한 구간 = stuck'); ax[i,0].set_ylabel('level')
fig.suptitle('stuck 센서 시계열 (전체기간)',fontsize=14); plt.tight_layout()
plt.savefig('./dataset/processed/viz/viz_stuck_timeseries.png',dpi=110); plt.close()
print('저장: viz_stuck_timeseries.png')

# ============ 2. 월별 결측 추이 ============
def monthly_missing(pattern, label):
    rows=[]
    for f in sorted(glob.glob(pattern)):
        df=pd.read_parquet(f,columns=['sensor_id','timestamp'])
        df['timestamp']=pd.to_datetime(df['timestamp'])
        t0,t1=df['timestamp'].min(),df['timestamp'].max()
        minutes=(t1-t0).total_seconds()/60+1
        nsens=df['sensor_id'].nunique()
        exp=nsens*minutes
        rows.append({'month':pd.to_datetime(t0.strftime('%Y-%m-01')),
                     'missing_pct':max(0,(1-len(df)/exp))*100,'n_sensors':nsens})
    return pd.DataFrame(rows).sort_values('month').assign(domain=label)

mm=pd.concat([monthly_missing(SEWER,'Sewer'), monthly_missing(ROAD,'Road')])
fig,ax=plt.subplots(1,1,figsize=(14,5))
for dom,g in mm.groupby('domain'):
    ax.plot(g['month'],g['missing_pct'],marker='o',label=dom)
ax.set_title('월별 결측 추이 (1분 그리드 대비)'); ax.set_xlabel('월'); ax.set_ylabel('결측 %'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('./dataset/processed/viz/viz_missing_trend.png',dpi=110); plt.close()
print('저장: viz_missing_trend.png')
print(mm.pivot(index='month',columns='domain',values='missing_pct').round(1).to_string())

# ============ 3. 통신상태별 분포 (하수만 — 도로엔 통신상태 컬럼 없음) ============
edges=np.linspace(0,12,120); 
from collections import defaultdict
hist=defaultdict(lambda:np.zeros(len(edges)-1)); cnt=defaultdict(int)
for f in sorted(glob.glob(SEWER)):
    df=pd.read_parquet(f,columns=['level','comm_status'])
    for cs,g in df.groupby('comm_status'):
        h,_=np.histogram(g['level'].dropna().clip(0,12),bins=edges); hist[cs]+=h; cnt[cs]+=len(g)
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].bar(range(len(cnt)),list(cnt.values()),color='slateblue'); ax[0].set_xticks(range(len(cnt))); ax[0].set_xticklabels(list(cnt.keys()),rotation=20)
ax[0].set_yscale('log'); ax[0].set_title('하수 통신상태별 건수(log)'); ax[0].set_ylabel('건수')
c=(edges[:-1]+edges[1:])/2
for cs,h in hist.items(): ax[1].plot(c,h,label=f'{cs} ({cnt[cs]:,})',lw=1)
ax[1].set_yscale('log'); ax[1].set_title('하수 통신상태별 level 분포(log y)'); ax[1].set_xlabel('level(m)'); ax[1].set_ylabel('건수(log)'); ax[1].legend()
fig.suptitle('통신상태별 분포 (하수, 전체기간)',fontsize=14); plt.tight_layout()
plt.savefig('./dataset/processed/viz/viz_commstatus.png',dpi=110); plt.close()
print('저장: viz_commstatus.png')
print('통신상태 건수:', dict(cnt))


한글폰트: NanumGothic
저장: viz_stuck_timeseries.png
저장: viz_missing_trend.png
domain      Road  Sewer
month                  
2022-01-01   NaN    3.8
2022-02-01   NaN    5.7
2022-03-01   NaN    5.9
2022-04-01   NaN    4.1
2022-05-01   NaN    4.6
2022-06-01   NaN    4.0
2022-07-01   NaN    4.5
2022-08-01   NaN    6.2
2022-09-01   NaN    4.8
2022-10-01   NaN   11.4
2022-11-01   NaN    8.8
2022-12-01   NaN    7.8
2023-01-01   NaN    8.3
2023-02-01   NaN   19.0
2023-03-01   NaN    8.2
2023-04-01   NaN    7.9
2023-05-01   NaN    6.5
2023-06-01   NaN    4.8
2023-07-01   NaN    4.1
2023-08-01   NaN    5.4
2023-09-01   NaN    2.7
2023-10-01   NaN    6.7
2023-11-01   NaN    6.9
2023-12-01   NaN   12.9
2024-01-01   7.6   15.5
2024-02-01  65.9   22.4
2024-03-01  14.0    3.7
2024-04-01  16.8    7.4
2024-05-01  37.2    6.1
2024-06-01  12.6    5.5
2024-07-01   7.5    2.6
2024-08-01  11.4    4.4
2024-09-01  23.7    6.0
2024-10-01  12.7    4.3
2024-11-01  23.1    5.6
2024-12-01  21.7    7.5
2025-01-01  11.

## Step 07. 인접행렬 생성 (Gaussian 가중치)

- `w = exp(-d / σ)`, σ=300m, threshold=0.1
- **sewer→road 엣지**: 상관 분석 결과 기반, 고립 도로 센서는 2km 이내 최근접 하수관과 fallback 연결
- **sewer→sewer 엣지**: 동일 배수구역 + 500m 이내 쌍


In [ ]:
import pandas as pd, numpy as np, os
from math import radians, sin, cos, sqrt, atan2
import pyarrow.parquet as pq

corr_df    = pd.read_parquet('./dataset/processed/cleaned/correlation_results.parquet', engine='pyarrow')
sewer_node = pd.read_parquet('./dataset/processed/cleaned/sewer_node.parquet', engine='pyarrow')
road_node  = pd.read_parquet('./dataset/processed/cleaned/road_node.parquet',  engine='pyarrow')
SIGMA, THRESHOLD = 300, 0.1

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000; d = lambda x: radians(x)
    dlat, dlon = d(lat2-lat1), d(lon2-lon1)
    a = sin(dlat/2)**2 + cos(d(lat1))*cos(d(lat2))*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# ── 기본 sewer→road 인접행렬 ─────────────────────────────────────────────────
adj = corr_df[['sewer_id','sewer_name','road_id','road_name','배수구역','자치구','distance_m']].copy()
adj['gauss_weight'] = np.exp(-adj['distance_m']/SIGMA)
adj = adj[adj['gauss_weight']>=THRESHOLD].reset_index(drop=True)
s_geo = sewer_node[['sensor_id','lat','lon']].rename(columns={'sensor_id':'sewer_id','lat':'s_lat','lon':'s_lon'})
r_geo = road_node[['sensor_id','lat','lon']].rename(columns={'sensor_id':'road_id','lat':'r_lat','lon':'r_lon'})
adj   = adj.merge(s_geo,on='sewer_id',how='left').merge(r_geo,on='road_id',how='left')
adj['edge_type'] = 'original'

# ── fallback 엣지 (고립 도로 센서 → 2km 이내 최근접 하수관) ─────────────────
isolated  = road_node[~road_node['sensor_id'].isin(adj['road_id'])].dropna(subset=['lat','lon'])
sewer_geo = sewer_node.dropna(subset=['lat','lon'])
new_rows  = []
for _,r in isolated.iterrows():
    best_d, best_s = 9999999, None
    for _,s in sewer_geo.iterrows():
        d = haversine(r['lat'],r['lon'],s['lat'],s['lon'])
        if d < best_d: best_d, best_s = d, s
    if best_d <= 2000 and best_s is not None:
        new_rows.append({'sewer_id':best_s['sensor_id'],'sewer_name':best_s['지점명'],
                         'road_id':r['sensor_id'],'road_name':r['지점명'],
                         '배수구역':best_s.get('배수구역',''),'자치구':best_s.get('자치구',''),
                         'distance_m':round(best_d,1),'gauss_weight':float(np.exp(-best_d/SIGMA)),
                         's_lat':best_s['lat'],'s_lon':best_s['lon'],
                         'r_lat':r['lat'],'r_lon':r['lon'],'edge_type':'fallback'})
adj_exp = pd.concat([adj, pd.DataFrame(new_rows)], ignore_index=True)

# ── sewer→sewer 엣지 (동일 배수구역 + 500m 이내) ─────────────────────────────
sewer_geo2 = sewer_geo[sewer_geo['배수구역'].notna()]
ids  = sewer_geo2['sensor_id'].tolist()
lats = sewer_geo2['lat'].tolist()
lons = sewer_geo2['lon'].tolist()
dsts = sewer_geo2['배수구역'].tolist()
ss_edges = []
for i in range(len(ids)):
    for j in range(i+1, len(ids)):
        if dsts[i] != dsts[j]: continue
        d = haversine(lats[i],lons[i],lats[j],lons[j])
        if d <= 500:
            ss_edges.append({'src_sewer_id':ids[i],'dst_sewer_id':ids[j],
                             'distance_m':round(d,1),'gauss_weight':float(np.exp(-d/SIGMA)),
                             '배수구역':dsts[i]})
df_ss = pd.DataFrame(ss_edges)

os.makedirs('./dataset/features/', exist_ok=True)
adj_exp.to_parquet('./dataset/features/adjacency_expanded.parquet', index=False)
df_ss.to_parquet('./dataset/features/sewer_sewer_edges.parquet', index=False)
print(f'sewer→road: {len(adj_exp)}개 / sewer→sewer: {len(df_ss)}개')


## Step 08. 공통 기간 분리 (2024-01 ~ 2025-08)

- 하수관로 (2022~2025) ∩ 도로노면 (2024~2026) → 교집합 기간만 사용
- 스트리밍 방식 (`ParquetWriter`)으로 메모리 절약


In [ ]:
import glob, os, gc
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq

OVERLAP_START = pd.Timestamp('2024-01-01')
OVERLAP_END   = pd.Timestamp('2025-08-31 23:59:59')
os.makedirs('./dataset/features/overlap/', exist_ok=True)

def stream_merge(files, out_path, schema, label):
    writer = pq.ParquetWriter(out_path, schema)
    total  = 0
    for f in sorted(files):
        df = pd.read_parquet(f, engine='pyarrow')
        df = df[(df['timestamp']>=OVERLAP_START)&(df['timestamp']<=OVERLAP_END)]
        if df.empty: continue
        for field in schema:
            if field.name not in df.columns: continue
            if pa.types.is_timestamp(field.type):
                df[field.name] = df[field.name].astype('datetime64[ns]')
            elif pa.types.is_floating(field.type):
                df[field.name] = pd.to_numeric(df[field.name], errors='coerce')
            elif pa.types.is_integer(field.type):
                df[field.name] = df[field.name].fillna(0).astype('int8')
        writer.write_table(pa.Table.from_pandas(df[schema.names], schema=schema, preserve_index=False))
        total += len(df); del df; gc.collect()
    writer.close()
    print(f'{label}: {total:,}행 → {os.path.getsize(out_path)/1e6:.0f} MB')

SEWER_FEAT_SCHEMA = pq.read_schema('./dataset/features/sewer/하수관로_수위_현황_202401.parquet')
ROAD_FEAT_SCHEMA  = pq.read_schema('./dataset/features/road/2024년 1월.parquet')

sewer_files = (sorted(glob.glob('./dataset/features/sewer/하수관로_수위_현황_2024*.parquet'))+
               sorted(glob.glob('./dataset/features/sewer/tv_swm_wal_mea_2025*.parquet')))
road_files  = (sorted(glob.glob('./dataset/features/road/2024년*.parquet'))+
               [f for f in sorted(glob.glob('./dataset/features/road/2025년*.parquet'))
                if any(f'2025년 {m}월' in f for m in ['1','2','3','4','5','6','7','8'])])

stream_merge(sewer_files, './dataset/features/overlap/sewer_overlap.parquet', SEWER_FEAT_SCHEMA, '하수관로')
stream_merge(road_files,  './dataset/features/overlap/road_overlap.parquet',  ROAD_FEAT_SCHEMA,  '도로노면')


## Step 09. 정규화

- **Train 데이터(2024년)에서만** 파라미터 계산 → data leakage 방지
- `level_norm`: 센서별 max 기준 min-max [0, 1]
- `level_diff_norm`: 센서별 z-score, clip [-5, 5]
- `hour_sin/cos`, `month_sin/cos`: cyclic 인코딩


In [ ]:
import pandas as pd, numpy as np, os, gc
import pyarrow as pa, pyarrow.parquet as pq, glob

sewer_node = pd.read_parquet('./dataset/processed/cleaned/sewer_node.parquet', engine='pyarrow')
road_node  = pd.read_parquet('./dataset/processed/cleaned/road_node.parquet',  engine='pyarrow')

# ── Train 기간(2024년) 데이터로만 파라미터 계산 ───────────────────────────────
train_s = pd.concat([pd.read_parquet(f, engine='pyarrow', columns=['sensor_id','level','level_diff'])
                     for f in sorted(glob.glob('./dataset/features/sewer/하수관로_수위_현황_2024*.parquet'))])
s_stats = train_s.groupby('sensor_id').agg(
    level_max  =('level','max'),
    level_p99  =('level', lambda x: x.quantile(0.99)),
    diff_mean  =('level_diff','mean'),
    diff_std   =('level_diff','std')).reset_index()
s_stats['phys_max']    = s_stats['sensor_id'].map(sewer_node.set_index('sensor_id')['max_level_m'].to_dict())
s_stats['level_scale'] = s_stats[['level_max','phys_max']].max(axis=1).fillna(s_stats['level_p99'])
s_stats.loc[s_stats['level_scale']<=0,'level_scale'] = 1.0
s_stats['diff_std'] = s_stats['diff_std'].fillna(1.0).replace(0,1.0)
del train_s

train_r = pd.concat([pd.read_parquet(f, engine='pyarrow', columns=['sensor_id','level','level_diff'])
                     for f in sorted(glob.glob('./dataset/features/road/2024년*.parquet'))])
r_stats = train_r.groupby('sensor_id').agg(
    level_max=('level','max'),
    diff_mean=('level_diff','mean'),
    diff_std =('level_diff','std')).reset_index()
r_stats['phys_max']    = r_stats['sensor_id'].map(road_node.set_index('sensor_id')['max_level_cm'].to_dict())
r_stats['level_scale'] = r_stats[['level_max','phys_max']].max(axis=1).fillna(96.0)
r_stats.loc[r_stats['level_scale']<=0,'level_scale'] = 96.0
r_stats['diff_std'] = r_stats['diff_std'].fillna(1.0).replace(0,1.0)
del train_r

s_stats.to_parquet('./dataset/features/overlap/sewer_norm_params.parquet', index=False)
r_stats.to_parquet('./dataset/features/overlap/road_norm_params.parquet',  index=False)
print(f'파라미터 저장: 하수 {len(s_stats)}개 / 도로 {len(r_stats)}개')


In [ ]:
# ── 정규화 적용 ────────────────────────────────────────────────────────────────
def apply_norm(df, scale_map, dmean_map, dstd_map, is_sewer=True):
    df['_sc'] = df['sensor_id'].map(scale_map).fillna(10.0 if is_sewer else 96.0)
    df['_dm'] = df['sensor_id'].map(dmean_map).fillna(0.0)
    df['_ds'] = df['sensor_id'].map(dstd_map).fillna(1.0)
    df['level_norm']      = (df['level']/df['_sc']).clip(0,1).astype('float32')
    df['level_diff_norm'] = ((df['level_diff']-df['_dm'])/df['_ds']).clip(-5,5).astype('float32')
    df.drop(columns=['_sc','_dm','_ds'], inplace=True)
    df['hour_sin']  = np.sin(2*np.pi*df['hour']/24).astype('float32')
    df['hour_cos']  = np.cos(2*np.pi*df['hour']/24).astype('float32')
    df['month_sin'] = np.sin(2*np.pi*df['month']/12).astype('float32')
    df['month_cos'] = np.cos(2*np.pi*df['month']/12).astype('float32')
    df['level']      = df['level'].astype('float32')
    df['level_diff'] = df['level_diff'].astype('float32')
    return df

s_params = pd.read_parquet('./dataset/features/overlap/sewer_norm_params.parquet')
r_params = pd.read_parquet('./dataset/features/overlap/road_norm_params.parquet')
s_scale  = s_params.set_index('sensor_id')['level_scale'].to_dict()
s_dmean  = s_params.set_index('sensor_id')['diff_mean'].to_dict()
s_dstd   = s_params.set_index('sensor_id')['diff_std'].to_dict()
r_scale  = r_params.set_index('sensor_id')['level_scale'].to_dict()
r_dmean  = r_params.set_index('sensor_id')['diff_mean'].to_dict()
r_dstd   = r_params.set_index('sensor_id')['diff_std'].to_dict()

for label, src, dst, is_sewer in [
    ('하수관로', './dataset/features/overlap/sewer_overlap.parquet',
              './dataset/features/overlap/sewer_normalized.parquet', True),
    ('도로노면', './dataset/features/overlap/road_overlap.parquet',
              './dataset/features/overlap/road_normalized.parquet',  False)]:
    df = pd.read_parquet(src, engine='pyarrow')
    scale_m, dm, ds = (s_scale,s_dmean,s_dstd) if is_sewer else (r_scale,r_dmean,r_dstd)
    df = apply_norm(df, scale_m, dm, ds, is_sewer)
    pq.write_table(pa.Table.from_pandas(df, preserve_index=False), dst)
    print(f'{label}: {len(df):,}행 → {os.path.getsize(dst)/1e6:.0f} MB')
    del df; gc.collect()


## Step 10. Train / Val / Test 시간 기반 분리

```
Train : 2024-01-01 ~ 2024-12-31  (12개월)
Val   : 2025-01-01 ~ 2025-05-31  (5개월)
Test  : 2025-06-01 ~ 2025-08-31  (3개월, 장마철)
```

> **NOTE**: Val/Test는 Train 정규화 파라미터(`sewer_norm_params.parquet`)를 그대로 적용합니다.


In [ ]:
import pandas as pd, os, gc
import pyarrow as pa, pyarrow.parquet as pq

SPLITS = {
    'train': (pd.Timestamp('2024-01-01'), pd.Timestamp('2024-12-31 23:59:59')),
    'val'  : (pd.Timestamp('2025-01-01'), pd.Timestamp('2025-05-31 23:59:59')),
    'test' : (pd.Timestamp('2025-06-01'), pd.Timestamp('2025-08-31 23:59:59')),
}
BASE = './dataset/features/overlap/'
for s in SPLITS:
    os.makedirs(BASE+s+'/', exist_ok=True)

for label in ['sewer', 'road']:
    src    = BASE+f'{label}_normalized.parquet'
    schema = pq.read_schema(src)
    df     = pd.read_parquet(src, engine='pyarrow')
    print(f'[{label}] {len(df):,}행 분리 중...')
    for split, (s, e) in SPLITS.items():
        out = BASE+f'{split}/{label}_{split}.parquet'
        sub = df[(df['timestamp']>=s)&(df['timestamp']<=e)].reset_index(drop=True)
        pq.write_table(pa.Table.from_pandas(sub, schema=schema, preserve_index=False), out)
        print(f'  {split}: {len(sub):,}행 / {os.path.getsize(out)/1e6:.0f} MB')
    del df; gc.collect()


## Step 11. GNN 설정 파일 생성

- 노드 인덱스 매핑 (`sensor_id` → `node_idx`)
- 클래스 가중치 계산 (Train 기준)
- Feature 목록, 시계열 설정, 강우 placeholder 기록


In [ ]:
import pandas as pd, numpy as np, json, os

BASE    = './dataset/features/'
adj_exp = pd.read_parquet(BASE+'adjacency_expanded.parquet')
ss      = pd.read_parquet(BASE+'sewer_sewer_edges.parquet')
r_norm  = pd.read_parquet(BASE+'overlap/road_normalized.parquet',  engine='pyarrow', columns=['sensor_id'])
s_norm  = pd.read_parquet(BASE+'overlap/sewer_normalized.parquet', engine='pyarrow', columns=['sensor_id'])

r_with_data = set(r_norm['sensor_id'].unique())
s_with_data = set(s_norm['sensor_id'].unique())

adj_valid = adj_exp[adj_exp['road_id'].isin(r_with_data)&adj_exp['sewer_id'].isin(s_with_data)].reset_index(drop=True)
ss_valid  = ss[ss['src_sewer_id'].isin(s_with_data)&ss['dst_sewer_id'].isin(s_with_data)].reset_index(drop=True)
adj_valid.to_parquet(BASE+'adjacency_expanded.parquet', index=False)
ss_valid.to_parquet(BASE+'sewer_sewer_edges.parquet', index=False)

sewer_ids = sorted((set(adj_valid['sewer_id'])|set(ss_valid['src_sewer_id']))&s_with_data)
road_ids  = sorted(set(adj_valid['road_id'])&r_with_data)
pd.DataFrame({'sensor_id':sewer_ids,'node_idx':range(len(sewer_ids)),'node_type':'sewer'}).to_parquet(BASE+'sewer_node_index.parquet',index=False)
pd.DataFrame({'sensor_id':road_ids, 'node_idx':range(len(road_ids)), 'node_type':'road'}).to_parquet(BASE+'road_node_index.parquet',index=False)

# 클래스 가중치 (Train 기준)
r_train = pd.read_parquet(BASE+'overlap/train/road_train.parquet', engine='pyarrow', columns=['flood_flag','flood_stage'])
n_pos   = int((r_train['flood_flag']==1).sum())
n_neg   = int((r_train['flood_flag']==0).sum())
stage_w = {int(k): round(len(r_train)/(5*v),2) for k,v in r_train['flood_stage'].value_counts().items()}

config = {
    'graph': {
        'sewer_road_edges':  len(adj_valid),
        'sewer_sewer_edges': len(ss_valid),
        'sewer_nodes': len(sewer_ids),
        'road_nodes':  len(road_ids),
        'sigma_m': 300, 'threshold': 0.1
    },
    'node_features': {
        'sewer': ['level_norm','level_diff_norm','fill_rate',
                  'hour_sin','hour_cos','month_sin','month_cos','is_weekend','rainfall_norm'],
        'road':  ['level_norm','level_diff_norm','flood_flag',
                  'hour_sin','hour_cos','month_sin','month_cos','is_weekend']
    },
    'target_features': {
        'road_regression':    'level_norm',
        'road_classification':'flood_flag',
        'road_multiclass':    'flood_stage'
    },
    'class_weights': {
        'binary': {'pos_weight':round(n_neg/n_pos,2),'n_positive':n_pos,'n_negative':n_neg,
                   'imbalance_ratio':f'1:{round(n_neg/n_pos)}'},
        'multiclass_stage': stage_w
    },
    'temporal': {'input_steps':6,'output_steps':18,'resolution_min':10},
    'train_val_test': {
        'train':['2024-01-01','2024-12-31'],
        'val':  ['2025-01-01','2025-05-31'],
        'test': ['2025-06-01','2025-08-31']
    },
    'future_extension': {
        'rainfall': {
            'feature_name': 'rainfall_norm',
            'description':  '시간당 강우량 (mm/hr), 최근접 AWS 기상 관측소',
            'status':       'active',
            'normalization':'min-max (0~100mm/hr 기준)',
            'position_in_sewer_features': 8,
            'note':         '서울시 강우관측망 48개소 통합 완료 — 하수관로 노드별 최근접 관측소 매핑'
        }
    }
}

with open(BASE+'gnn_config.json','w',encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print('=== GNN 데이터셋 준비 완료 ===')
print(f'하수관 노드: {len(sewer_ids)}개 / 도로 노드: {len(road_ids)}개')
print(f'sewer→road: {len(adj_valid)}개 / sewer→sewer: {len(ss_valid)}개')
print(f'클래스 불균형: 1:{round(n_neg/n_pos)}')
print('gnn_config.json 저장 완료')


## 최종 검증

전처리 결과 파일 목록과 크기를 확인합니다.

In [ ]:
import os

BASE = './dataset/features/'
checks = [
    (BASE+'adjacency_expanded.parquet',          'sewer→road 엣지'),
    (BASE+'sewer_sewer_edges.parquet',            'sewer→sewer 엣지'),
    (BASE+'sewer_node_index.parquet',             '하수 노드 인덱스'),
    (BASE+'road_node_index.parquet',              '도로 노드 인덱스'),
    (BASE+'gnn_config.json',                      'GNN 설정'),
    (BASE+'overlap/sewer_normalized.parquet',     '하수 정규화'),
    (BASE+'overlap/road_normalized.parquet',      '도로 정규화'),
    (BASE+'overlap/train/sewer_train.parquet',    '하수 Train'),
    (BASE+'overlap/train/road_train.parquet',     '도로 Train'),
    (BASE+'overlap/val/sewer_val.parquet',        '하수 Val'),
    (BASE+'overlap/val/road_val.parquet',         '도로 Val'),
    (BASE+'overlap/test/sewer_test.parquet',      '하수 Test'),
    (BASE+'overlap/test/road_test.parquet',       '도로 Test'),
    (BASE+'overlap/sewer_norm_params.parquet',    '하수 정규화 파라미터'),
    (BASE+'overlap/road_norm_params.parquet',     '도로 정규화 파라미터'),
]
all_ok = True
for path, desc in checks:
    ok   = os.path.exists(path)
    size = os.path.getsize(path)/1e6 if ok else 0
    print(f'  {"OK" if ok else "MISSING":<8} {desc:<28} {size:>7.1f} MB')
    if not ok: all_ok = False
print(f'\n{"전처리 완료 — 모델 학습 준비 완료" if all_ok else "미완료 항목 확인 필요"}')
